# CELL 01 — GPU Check


In [ ]:
import subprocess
import sys

# Check GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# Check CUDA
import torch
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name         : {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Total VRAM       : {total_mem:.1f} GB")

    # Warn if not A100
    gpu_name = torch.cuda.get_device_name(0)
    if "A100" in gpu_name:
        print(f"\n✅ PERFECT — A100 detected. We have full power!")
    elif "V100" in gpu_name:
        print(f"\n🟡 V100 detected — workable but Qwen2.5-14B")
        print(f"   may be tight. Consider Qwen2.5-7B instead.")
    else:
        print(f"\n⚠️  Non-A100 GPU detected.")
        print(f"   Recommend switching to A100 in Colab Pro.")
else:
    print("\n❌ No GPU detected!")
    print("   Go to: Runtime → Change Runtime Type → A100 GPU")
    sys.exit("Please enable GPU and restart.")

# Check RAM
import psutil
ram = psutil.virtual_memory()
print(f"\nSystem RAM       : {ram.total / 1024**3:.1f} GB")
print(f"Available RAM    : {ram.available / 1024**3:.1f} GB")

# Check Python
print(f"\nPython version   : {sys.version.split()[0]}")
print("\n✅ Cell 01 complete — environment looks good!")


# CELL 02 — Install All Dependencies


In [ ]:
import subprocess
import sys

print("📦 Installing all RAG103 dependencies...")
print("⏳ This will take 3-5 minutes. Grab a coffee ☕\n")

packages = [
    # Fix numpy first
    "numpy==1.26.4",

    # Core ML
    "torch",
    "transformers>=4.45.0",
    "accelerate>=0.26.0",
    "bitsandbytes>=0.43.0",

    # Embeddings & Retrieval
    "sentence-transformers",
    "faiss-cpu",
    "rank-bm25",

    # Chunking
    "spacy>=3.7.0",

    # Agentic Framework
    "langgraph>=0.2.0",
    "langchain>=0.3.0",
    "langchain-core>=0.3.0",
    "langchain-community>=0.3.0",

    # Data
    "datasets>=2.20.0",
    "pandas",

    # Evaluation
    "rouge-score",
    "nltk",
    "scikit-learn",

    # API & UI
    "fastapi",
    "uvicorn",
    "pyngrok",
    "nest-asyncio",
    "gradio>=4.0.0",
    "pydantic>=2.0.0",

    # Utilities
    "tqdm",
    "psutil",
    "python-dotenv",
    "rich",
]

# Install all at once
install_cmd = [sys.executable, "-m", "pip", "install", "-q", "-U"] + packages
result = subprocess.run(install_cmd, capture_output=True, text=True)

if result.returncode == 0:
    print("✅ All packages installed successfully!\n")
else:
    print("⚠️  Some warnings during install (usually safe):")
    errors = [l for l in result.stderr.split('\n')
              if 'ERROR' in l.upper() and 'warning' not in l.lower()]
    if errors:
        for e in errors:
            print(f"   {e}")
    else:
        print("   Only warnings detected — safe to continue.\n")

# Install sentence-transformers separately
print("📥 Installing sentence-transformers...")
st_result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "sentence-transformers"],
    capture_output=True, text=True
)
print("✅ sentence-transformers ready!\n" if st_result.returncode == 0
      else f"❌ Failed: {st_result.stderr[-200:]}")

# Install spaCy English model
print("📥 Installing spaCy English model...")
spacy_result = subprocess.run(
    [sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
    capture_output=True, text=True
)
if spacy_result.returncode == 0:
    print("✅ spaCy model ready!\n")
else:
    print("⚠️  spaCy model install output:")
    print(spacy_result.stdout[-500:] if spacy_result.stdout else "No output")

# Install NLTK data
print("📥 Installing NLTK data...")
import importlib
import nltk
importlib.reload(nltk)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
print("✅ NLTK data ready!\n")

# Verify critical packages
print("🔍 Verifying critical installations:")
critical = {
    "numpy"                : "numpy",
    "torch"                : "torch",
    "transformers"         : "transformers",
    "sentence_transformers": "sentence_transformers",
    "faiss"                : "faiss",
    "langgraph"            : "langgraph",
    "langchain"            : "langchain",
    "spacy"                : "spacy",
    "datasets"             : "datasets",
    "gradio"               : "gradio",
    "fastapi"              : "fastapi",
    "rank_bm25"            : "rank_bm25",
}

all_good = True
for display_name, import_name in critical.items():
    try:
        mod     = __import__(import_name)
        version = getattr(mod, '__version__', 'installed')
        print(f"   ✅ {display_name:<25} {version}")
    except ImportError:
        print(f"   ❌ {display_name:<25} NOT FOUND")
        all_good = False

if all_good:
    print("\n✅ Cell 02 complete — all dependencies verified!")
else:
    print("\n❌ Some packages missing — do not proceed!")
    print("   Re-run this cell before continuing.")

# CELL 03 — All Imports


In [ ]:
# CELL 03 — All Imports

import os
import re
import json
import math
import time
import random
import pickle
import threading
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from typing import TypedDict, Annotated, List, Optional, Literal

# PyTorch
import torch

# Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

# Sentence Transformers
from sentence_transformers import SentenceTransformer, CrossEncoder

# FAISS
import faiss

# BM25
from rank_bm25 import BM25Okapi

# spaCy
import spacy

# Datasets
from datasets import load_dataset

# LangGraph
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# LangChain
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Evaluation
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
import nltk

# API & UI
import uvicorn
import nest_asyncio
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import gradio as gr
from pyngrok import ngrok

# Utilities
import psutil

print(f"✅ All imports successful!")
print(f"\n📊 Environment Summary:")
print(f"   PyTorch        : {torch.__version__}")
print(f"   CUDA Available : {torch.cuda.is_available()}")
print(f"   GPU            : {torch.cuda.get_device_name(0)}")
print(f"   VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"   System RAM     : {psutil.virtual_memory().total / 1024**3:.1f} GB")
print(f"\n✅ Cell 03 complete — ready to build RAG103!")

# CELL 04 — Master Configuration


In [ ]:
# CELL 04 — Master Configuration

# ── Data ──────────────────────────────────────────────────────────────────────
MAX_SCOTUS_CASES     = 5000      # SCOTUS cases to load
MAX_CUAD_CONTRACTS   = 510       # CUAD contracts (full dataset)

# ── Chunking ──────────────────────────────────────────────────────────────────
MIN_CHUNK_SENTENCES  = 3         # Minimum sentences per chunk
MAX_CHUNK_SENTENCES  = 8         # Maximum sentences per chunk
SIMILARITY_THRESHOLD = 0.75      # Semantic similarity to group sentences
MAX_CHUNK_CHARS      = 1500      # Hard cap on chunk character length

# ── Retrieval ─────────────────────────────────────────────────────────────────
TOPK_DENSE           = 60        # Dense candidates per corpus
TOPK_BM25            = 60        # BM25 candidates per corpus
TOPK_FUSED           = 100        # After RRF fusion
TOPK_RERANK          = 10        # After cross-encoder reranking
RRF_K                = 60        # RRF damping constant (standard value)

# ── Retrieval Score Gates ─────────────────────────────────────────────────────
RERANK_THRESHOLD     = 0.60      # Below this → reflection agent fires
CORRECT_THRESHOLD    = 0.65      # Above this → chunk is CORRECT
AMBIGUOUS_LOW        = 0.40      # Below this → chunk is INCORRECT

# ── Agent Loop Limits ─────────────────────────────────────────────────────────
MAX_HOPS             = 3         # Maximum retrieval hops
MAX_REWRITES         = 2         # Maximum query rewrites
MAX_REGENERATIONS    = 1         # Maximum answer regenerations

# ── Generation ────────────────────────────────────────────────────────────────
GEN_MODEL            = "Qwen/Qwen2.5-14B-Instruct"   # upgraded from 7B
MAX_INPUT_TOKENS     = 4096      # Context window for generation
MAX_NEW_TOKENS       = 512       # Maximum tokens to generate

# ── Embedder & Reranker ───────────────────────────────────────────────────────
EMBEDDER_MODEL       = "BAAI/bge-m3"
RERANKER_MODEL       = "BAAI/bge-reranker-large"
EMBEDDING_DIM        = 1024      # BGE-M3 output dimension
EMBED_BATCH_SIZE     = 128       # Batch size for encoding

# ── Classifier ────────────────────────────────────────────────────────────────
CLASSIFIER_THRESHOLD = 0.40      # Minimum similarity to assign intent

# ── Evaluation ────────────────────────────────────────────────────────────────
EVAL_RETRIEVAL_SAMPLES = 100     # LLM-generated Q&A pairs for retrieval eval
EVAL_K_LIST            = (1, 3, 5, 10)  # K values for Recall@K and MRR@K
CUAD_EVAL_SAMPLES      = 50      # CUAD gold Q&A pairs for eval

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_SAVE_DIR       = "/content/drive/MyDrive/RAG103"

# ── Seeds ─────────────────────────────────────────────────────────────────────
RANDOM_SEED          = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("✅ Master configuration loaded.")
print(f"\n📋 Key Settings:")
print(f"   Generator Model  : {GEN_MODEL}")
print(f"   Embedder Model   : {EMBEDDER_MODEL}")
print(f"   Reranker Model   : {RERANKER_MODEL}")
print(f"   Max SCOTUS Cases : {MAX_SCOTUS_CASES}")
print(f"   Max CUAD Docs    : {MAX_CUAD_CONTRACTS}")
print(f"   Max Hops         : {MAX_HOPS}")
print(f"   Max Rewrites     : {MAX_REWRITES}")
print(f"   RRF K            : {RRF_K}")
print(f"\n✅ Cell 04 complete — configuration ready!")


#CELL 05 — 📚 Load SCOTUS Dataset


In [ ]:
print("📚 Loading SCOTUS dataset...")
print(f"   Source  : lex_glue/scotus")
print(f"   Max cases: {MAX_SCOTUS_CASES}\n")

# Load from HuggingFace
ds = load_dataset("lex_glue", "scotus", split="train")

# Filter and collect
scotus_docs = []
for row in tqdm(ds, desc="Processing SCOTUS cases"):
    text = row.get("text", "").strip()

    # Skip very short cases (not useful for retrieval)
    if len(text) < 1500:
        continue

    scotus_docs.append({
        "text"     : text,
        "source"   : "scotus",
        "doc_id"   : len(scotus_docs),
        "label"    : row.get("label", -1),
    })

    if len(scotus_docs) >= MAX_SCOTUS_CASES:
        break

# Stats
avg_len = sum(len(d["text"]) for d in scotus_docs) / len(scotus_docs)
min_len = min(len(d["text"]) for d in scotus_docs)
max_len = max(len(d["text"]) for d in scotus_docs)

print(f"\n📊 SCOTUS Dataset Stats:")
print(f"   Total cases loaded : {len(scotus_docs)}")
print(f"   Avg document length: {avg_len:,.0f} chars")
print(f"   Min document length: {min_len:,.0f} chars")
print(f"   Max document length: {max_len:,.0f} chars")
print(f"\n📋 Sample case (first 300 chars):")
print(f"   {scotus_docs[0]['text'][:300]}...")
print(f"\n✅ Cell 05 complete — SCOTUS dataset ready!")


# CELL 06 — 📜 Load CUAD Dataset

In [ ]:
# CELL 06 — Find Correct CUAD on HuggingFace

from huggingface_hub import list_datasets

print("🔍 Searching HuggingFace for CUAD datasets...\n")

results = list(list_datasets(search="cuad", limit=20))
for r in results:
    print(f"   → {r.id}")

In [ ]:
# CELL 06 — Find Correct CUAD on HuggingFace
print("📜 Loading CUAD dataset...")
print(f"   Max contracts: {MAX_CUAD_CONTRACTS}\n")

sources = [
    "chenghao/cuad_qa",
    "yogesh0502/cuad_v1",
    "kenlevine/CUAD",
    "alex-apostolo/filtered-cuad",
]

cuad_raw = None
for source in sources:
    try:
        print(f"   Trying: {source}...")
        cuad_raw = load_dataset(
            source,
            split="train",
            verification_mode="no_checks"
        )
        print(f"   ✅ Loaded from: {source}")
        print(f"   📋 Rows   : {len(cuad_raw)}")
        print(f"   📋 Columns: {cuad_raw.column_names}\n")
        break
    except Exception as e:
        print(f"   ❌ {str(e)[:80]}\n")

if cuad_raw is None:
    print("❌ All sources failed — paste output and we'll find another way.")
else:
    print("📋 Sample row structure:")
    sample = cuad_raw[0]
    for key, val in sample.items():
        print(f"   {key:<15} : {str(val)[:80]}")
    print()

    seen_titles  = set()
    cuad_docs    = []
    cuad_qa_gold = []

    for row in tqdm(cuad_raw, desc="Processing CUAD contracts"):

        context  = row.get("context",  "").strip()
        title    = row.get("title",    "").strip()
        question = row.get("question", "").strip()
        answers  = row.get("answers",  {})

        if not context or len(context) < 500:
            continue

        if title not in seen_titles:
            seen_titles.add(title)
            cuad_docs.append({
                "text"  : context,
                "source": "cuad",
                "doc_id": len(cuad_docs),
                "title" : title,
            })

        if isinstance(answers, dict):
            answer_texts = answers.get("text", [])
        elif isinstance(answers, list):
            answer_texts = [a.get("text", "") if isinstance(a, dict) else a for a in answers]
        else:
            answer_texts = []

        if question and answer_texts and len(str(answer_texts[0])) > 10:
            cuad_qa_gold.append({
                "question"      : question,
                "answer"        : str(answer_texts[0]),
                "contract_title": title,
                # ✅ FIX: increased from 2000 → 4000 so gold context
                # is not truncated, improving semantic similarity scores
                "context"       : context[:4000],
            })

        if len(cuad_docs) >= MAX_CUAD_CONTRACTS:
            break

    if not cuad_docs:
        print("❌ No documents loaded!")
        print(f"   Sample row: {str(cuad_raw[0])[:500]}")
    else:
        avg_len = sum(len(d["text"]) for d in cuad_docs) / len(cuad_docs)
        min_len = min(len(d["text"]) for d in cuad_docs)
        max_len = max(len(d["text"]) for d in cuad_docs)

        print(f"\n📊 CUAD Dataset Stats:")
        print(f"   Unique contracts loaded : {len(cuad_docs)}")
        print(f"   Gold Q&A pairs collected: {len(cuad_qa_gold)}")
        print(f"   Avg contract length     : {avg_len:,.0f} chars")
        print(f"   Min contract length     : {min_len:,.0f} chars")
        print(f"   Max contract length     : {max_len:,.0f} chars")
        print(f"   Gold context cap        : 4000 chars")

        print(f"\n📋 Sample contract title:")
        print(f"   {cuad_docs[0]['title']}")

        if cuad_qa_gold:
            print(f"\n📋 Sample gold Q&A pair:")
            print(f"   Q: {cuad_qa_gold[0]['question']}")
            print(f"   A: {cuad_qa_gold[0]['answer'][:200]}...")

        print(f"\n✅ Cell 06 complete — CUAD dataset ready!")

# CELL 07 — Unified Data Schema


In [ ]:
print("🗂️  Building unified data schema...")
print(f"   SCOTUS docs : {len(scotus_docs)}")
print(f"   CUAD docs   : {len(cuad_docs)}\n")

def clean_text(text: str) -> str:
    """
    Clean raw legal text:
    - Collapse whitespace
    - Remove null bytes
    - Normalize unicode dashes
    - Strip leading/trailing whitespace
    """
    text = text.replace("\x00", " ")
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2014", "-").replace("\u2013", "-")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# ── Build unified corpus
# Schema:
# {
#   "doc_id"  : int       → unique ID across both corpora
#   "text"    : str       → cleaned document text
#   "source"  : str       → "scotus" or "cuad"
#   "title"   : str       → case name or contract title
#   "label"   : int       → SCOTUS label (-1 for CUAD)
#   "char_len": int       → character length
#   "word_len": int       → word count
# }

unified_corpus = []
doc_id_counter = 0

# Add SCOTUS docs
print("📚 Processing SCOTUS docs...")
for doc in tqdm(scotus_docs, desc="   SCOTUS"):
    cleaned = clean_text(doc["text"])
    unified_corpus.append({
        "doc_id"  : doc_id_counter,
        "text"    : cleaned,
        "source"  : "scotus",
        "title"   : f"SCOTUS_case_{doc['doc_id']}",
        "label"   : doc.get("label", -1),
        "char_len": len(cleaned),
        "word_len": len(cleaned.split()),
    })
    doc_id_counter += 1

# Add CUAD docs
print("📜 Processing CUAD docs...")
for doc in tqdm(cuad_docs, desc="   CUAD  "):
    cleaned = clean_text(doc["text"])
    unified_corpus.append({
        "doc_id"  : doc_id_counter,
        "text"    : cleaned,
        "source"  : "cuad",
        "title"   : doc.get("title", f"CUAD_contract_{doc['doc_id']}"),
        "label"   : -1,
        "char_len": len(cleaned),
        "word_len": len(cleaned.split()),
    })
    doc_id_counter += 1

# ── Separate corpus references ────────────────────────────────────────────────
# Keep separate lists for corpus-specific retrieval
scotus_corpus = [d for d in unified_corpus if d["source"] == "scotus"]
cuad_corpus   = [d for d in unified_corpus if d["source"] == "cuad"]

# ── Stats ─────────────────────────────────────────────────────────────────────
total_chars = sum(d["char_len"] for d in unified_corpus)
total_words = sum(d["word_len"] for d in unified_corpus)

print(f"\n📊 Unified Corpus Stats:")
print(f"   Total documents      : {len(unified_corpus):,}")
print(f"   ├── SCOTUS           : {len(scotus_corpus):,}")
print(f"   └── CUAD             : {len(cuad_corpus):,}")
print(f"   Total characters     : {total_chars:,}")
print(f"   Total words          : {total_words:,}")
print(f"   Total pages (approx) : {total_words // 250:,}")

print(f"\n📋 Schema verification (first SCOTUS doc):")
sample_s = scotus_corpus[0]
for k, v in sample_s.items():
    print(f"   {k:<10} : {str(v)[:60]}")

print(f"\n📋 Schema verification (first CUAD doc):")
sample_c = cuad_corpus[0]
for k, v in sample_c.items():
    print(f"   {k:<10} : {str(v)[:60]}")

print(f"\n✅ Cell 07 complete — unified corpus ready!")

# CELL 08 — Hybrid Legal Chunker


In [ ]:
print("✂️  Building hybrid legal chunker...")

# ── Load spaCy ────────────────────────────────────────────────────────────────
print("📥 Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")
nlp.max_length = 2_000_000
print("✅ spaCy ready\n")

# ── Legal structure markers ───────────────────────────────────────────────────
# SCOTUS markers
SCOTUS_MARKERS = re.compile(
    r"\n\s*("
    r"HELD[:.]|FACTS[:.]|OPINION[:.]|DISSENT[:.]|CONCUR[A-Z]*[:.]|"
    r"SYLLABUS[:.]|JUDGMENT[:.]|REVERSED|AFFIRMED|REMANDED|"
    r"I\.|II\.|III\.|IV\.|V\.|VI\.|VII\.|VIII\.|IX\.|X\.|"
    r"[A-Z]\.\s+[A-Z]|"                    # Section headers like "A. Background"
    r"MR\. JUSTICE|MR\. CHIEF JUSTICE|"
    r"Justice [A-Z][a-z]+\s+delivered|"
    r"The (Court|judgment|opinion)"
    r")",
    re.IGNORECASE
)

# CUAD markers
CUAD_MARKERS = re.compile(
    r"\n\s*("
    r"ARTICLE\s+[IVXLC\d]+|"
    r"SECTION\s+\d+|"
    r"Section\s+\d+\.|"
    r"\d+\.\d*\s+[A-Z]|"                   # Numbered clauses like "1.1 Definitions"
    r"WHEREAS|NOW THEREFORE|"
    r"IN WITNESS WHEREOF|"
    r"RECITALS|DEFINITIONS|TERM|"
    r"TERMINATION|CONFIDENTIALITY|"
    r"GOVERNING LAW|INDEMNIFICATION"
    r")",
    re.IGNORECASE
)

# ── Core chunking functions ───────────────────────────────────────────────────

def split_by_structure(text: str, source: str) -> list:
    """
    Step 1: Split document at legal structure markers.
    Returns list of sections.
    """
    marker = SCOTUS_MARKERS if source == "scotus" else CUAD_MARKERS
    sections = marker.split(text)
    sections = [s.strip() for s in sections if s and len(s.strip()) > 100]
    return sections if sections else [text]


def split_by_paragraphs(text: str) -> list:
    """
    Step 2: Split at paragraph boundaries (\n\n).
    """
    paras = re.split(r"\n\s*\n", text)
    paras = [p.strip() for p in paras if len(p.strip()) > 100]
    return paras if paras else [text]


def split_by_sentences(text: str) -> list:
    """
    Step 3: Split at sentence boundaries using spaCy.
    Groups sentences into chunks respecting MAX_CHUNK_CHARS.
    """
    if len(text) > 500_000:
        text = text[:500_000]

    doc       = nlp(text)
    sentences = [s.text.strip() for s in doc.sents if len(s.text.strip()) > 20]

    if not sentences:
        return [text]

    chunks        = []
    current_chunk = []
    current_len   = 0

    for sent in sentences:
        if current_len + len(sent) > MAX_CHUNK_CHARS and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sent]
            current_len   = len(sent)
        else:
            current_chunk.append(sent)
            current_len += len(sent)

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks if chunks else [text]


def hybrid_legal_chunk(text: str, source: str) -> list:
    """
    Hybrid Legal Chunker Pipeline:

    Step 1 → Split at legal structure markers
             (HELD:, ARTICLE:, SECTION:, etc.)

    Step 2 → If section still too long
             → split at paragraph breaks (\n\n)

    Step 3 → If paragraph still too long
             → split at sentence boundaries (spaCy)

    Result → Chunks that respect legal document
             structure at every level ✅
    """
    final_chunks = []

    # Step 1: Legal structure split
    sections = split_by_structure(text, source)

    for section in sections:
        if len(section) <= MAX_CHUNK_CHARS:
            # Section is good size → keep as chunk
            final_chunks.append(section)
        else:
            # Step 2: Paragraph split
            paragraphs = split_by_paragraphs(section)
            for para in paragraphs:
                if len(para) <= MAX_CHUNK_CHARS:
                    # Paragraph is good size → keep as chunk
                    final_chunks.append(para)
                else:
                    # Step 3: Sentence split
                    sent_chunks = split_by_sentences(para)
                    final_chunks.extend(sent_chunks)

    # Final cleanup
    final_chunks = [
        c.strip() for c in final_chunks
        if len(c.strip()) > 100
    ]

    return final_chunks


# ── Quick test ────────────────────────────────────────────────────────────────
print("🧪 Testing on SCOTUS sample...")
scotus_sample  = scotus_corpus[0]["text"]
scotus_test    = hybrid_legal_chunk(scotus_sample, "scotus")
print(f"   Doc length      : {len(scotus_sample):,} chars")
print(f"   Chunks produced : {len(scotus_test)}")
print(f"   Avg chunk size  : {np.mean([len(c) for c in scotus_test]):,.0f} chars")
print(f"   Sample chunk 1  : {scotus_test[0][:150]}...")
print(f"   Sample chunk 2  : {scotus_test[1][:150]}...")

print("\n🧪 Testing on CUAD sample...")
cuad_sample = cuad_corpus[0]["text"]
cuad_test   = hybrid_legal_chunk(cuad_sample, "cuad")
print(f"   Doc length      : {len(cuad_sample):,} chars")
print(f"   Chunks produced : {len(cuad_test)}")
print(f"   Avg chunk size  : {np.mean([len(c) for c in cuad_test]):,.0f} chars")
print(f"   Sample chunk 1  : {cuad_test[0][:150]}...")
print(f"   Sample chunk 2  : {cuad_test[1][:150]}...")

print(f"\n✅ Cell 08 complete — hybrid legal chunker ready!")

# CELL 09 — Chunk Both Corpora


In [ ]:
print("📦 Chunking all documents with hybrid legal chunker...")
print(f"   SCOTUS : {len(scotus_corpus)} docs")
print(f"   CUAD   : {len(cuad_corpus)} docs")
print(f"   ⏳ Expected time: 5-10 minutes\n")

# ── Chunk SCOTUS ──────────────────────────────────────────────────────────────
scotus_chunks = []

for doc in tqdm(scotus_corpus, desc="Chunking SCOTUS"):
    doc_chunks = hybrid_legal_chunk(doc["text"], "scotus")
    for chunk_idx, chunk_text in enumerate(doc_chunks):
        scotus_chunks.append({
            "chunk_id" : len(scotus_chunks),
            "doc_id"   : doc["doc_id"],
            "chunk_idx": chunk_idx,
            "text"     : chunk_text,
            "source"   : "scotus",
            "title"    : doc["title"],
            "label"    : doc["label"],
            "char_len" : len(chunk_text),
        })

print(f"\n✅ SCOTUS chunking complete!")
print(f"   Total SCOTUS chunks : {len(scotus_chunks):,}")
print(f"   Avg chunks per doc  : {len(scotus_chunks)/len(scotus_corpus):.1f}")
print(f"   Avg chunk size      : {np.mean([c['char_len'] for c in scotus_chunks]):,.0f} chars")

# ── Chunk CUAD ────────────────────────────────────────────────────────────────
cuad_chunks = []

for doc in tqdm(cuad_corpus, desc="Chunking CUAD  "):
    doc_chunks = hybrid_legal_chunk(doc["text"], "cuad")
    for chunk_idx, chunk_text in enumerate(doc_chunks):
        cuad_chunks.append({
            "chunk_id" : len(cuad_chunks),
            "doc_id"   : doc["doc_id"],
            "chunk_idx": chunk_idx,
            "text"     : chunk_text,
            "source"   : "cuad",
            "title"    : doc["title"],
            "label"    : -1,
            "char_len" : len(chunk_text),
        })

print(f"\n✅ CUAD chunking complete!")
print(f"   Total CUAD chunks   : {len(cuad_chunks):,}")
print(f"   Avg chunks per doc  : {len(cuad_chunks)/len(cuad_corpus):.1f}")
print(f"   Avg chunk size      : {np.mean([c['char_len'] for c in cuad_chunks]):,.0f} chars")

# ── Combined stats ────────────────────────────────────────────────────────────
all_chunks  = scotus_chunks + cuad_chunks
all_lengths = [c["char_len"] for c in all_chunks]

print(f"\n📊 Combined Chunking Stats:")
print(f"   Total chunks        : {len(all_chunks):,}")
print(f"   ├── SCOTUS chunks   : {len(scotus_chunks):,}")
print(f"   └── CUAD chunks     : {len(cuad_chunks):,}")
print(f"   Avg chunk size      : {np.mean(all_lengths):,.0f} chars")
print(f"   Min chunk size      : {np.min(all_lengths):,.0f} chars")
print(f"   Max chunk size      : {np.max(all_lengths):,.0f} chars")
print(f"   Std deviation       : {np.std(all_lengths):,.0f} chars")

print(f"\n✅ Cell 09 complete — all chunks ready!")

# CELL 10 — Save Everything to Google Drive


In [ ]:
import os
import pickle
from google.colab import drive

print("💾 Saving all data to Google Drive...\n")

# ── Mount Drive ───────────────────────────────────────────────────────────────
drive.mount("/content/drive")
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"✅ Drive mounted → {DRIVE_SAVE_DIR}\n")

# ── Save function ─────────────────────────────────────────────────────────────
def save_pickle(obj, filename: str):
    path   = os.path.join(DRIVE_SAVE_DIR, filename)
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    size = os.path.getsize(path) / 1024**2
    print(f"   ✅ {filename:<30} {size:.1f} MB")

# ── Save everything ───────────────────────────────────────────────────────────
print("📦 Saving chunks:")
save_pickle(scotus_chunks,    "scotus_chunks.pkl")
save_pickle(cuad_chunks,      "cuad_chunks.pkl")

print("\n📚 Saving corpora:")
save_pickle(scotus_corpus,    "scotus_corpus.pkl")
save_pickle(cuad_corpus,      "cuad_corpus.pkl")
save_pickle(unified_corpus,   "unified_corpus.pkl")

print("\n🎯 Saving evaluation data:")
save_pickle(cuad_qa_gold,     "cuad_qa_gold.pkl")

# ── Save metadata summary ─────────────────────────────────────────────────────
metadata = {
    "total_scotus_docs"   : len(scotus_corpus),
    "total_cuad_docs"     : len(cuad_corpus),
    "total_scotus_chunks" : len(scotus_chunks),
    "total_cuad_chunks"   : len(cuad_chunks),
    "total_gold_qa_pairs" : len(cuad_qa_gold),
    "chunker"             : "hybrid_legal",
    "embedder_model"      : EMBEDDER_MODEL,
    "generator_model"     : GEN_MODEL,
}
save_pickle(metadata, "metadata.pkl")

# ── Verify all files saved ────────────────────────────────────────────────────
print(f"\n🔍 Verifying saved files:")
total_size = 0
for f in os.listdir(DRIVE_SAVE_DIR):
    if f.endswith(".pkl"):
        fpath = os.path.join(DRIVE_SAVE_DIR, f)
        fsize = os.path.getsize(fpath) / 1024**2
        total_size += fsize
        print(f"   ✅ {f:<35} {fsize:.1f} MB")

print(f"\n   Total saved         : {total_size:.1f} MB")

print(f"""
{'='*55}
💾 Save Complete!
{'='*55}
📍 Location : {DRIVE_SAVE_DIR}

Next session load order:
   1. Run Cells 01-04 (setup)
   2. Run Cell 10b   (load from Drive)
   3. Skip Cells 05-09 entirely
   4. Continue from Cell 11
{'='*55}
""")
print("✅ Cell 10 complete!")

# CELL 10b — 📂 Load From Drive


In [ ]:
import os
import pickle
import time
from google.colab import drive

print("📂 Loading all data from Google Drive...\n")

# ── Mount Drive ───────────────────────────────────────────────────────────────
drive.flush_and_unmount()
time.sleep(2)
drive.mount("/content/drive", force_remount=True)
time.sleep(3)
print()

# ── Load function with retry ──────────────────────────────────────────────────
def load_pickle(filename: str, retries: int = 3):
    path = os.path.join(DRIVE_SAVE_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"❌ {filename} not found in Drive!")

    for attempt in range(retries):
        try:
            with open(path, "rb") as f:
                obj = pickle.load(f)
            size = os.path.getsize(path) / 1024**2
            print(f"   ✅ {filename:<30} {size:.1f} MB")
            return obj
        except OSError as e:
            if attempt < retries - 1:
                print(f"   ⚠️  Attempt {attempt+1} failed — retrying in 3s...")
                time.sleep(3)
            else:
                raise OSError(f"❌ Failed to load {filename} after {retries} attempts: {e}")

# ── Load everything ───────────────────────────────────────────────────────────
print("📦 Loading chunks:")
scotus_chunks  = load_pickle("scotus_chunks.pkl")
cuad_chunks    = load_pickle("cuad_chunks.pkl")

print("\n📚 Loading corpora:")
scotus_corpus  = load_pickle("scotus_corpus.pkl")
cuad_corpus    = load_pickle("cuad_corpus.pkl")
unified_corpus = load_pickle("unified_corpus.pkl")

print("\n🎯 Loading evaluation data:")
cuad_qa_gold   = load_pickle("cuad_qa_gold.pkl")

print("\n📋 Loading metadata:")
metadata       = load_pickle("metadata.pkl")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n📊 Loaded Summary:")
print(f"   SCOTUS docs     : {len(scotus_corpus):,}")
print(f"   CUAD docs       : {len(cuad_corpus):,}")
print(f"   SCOTUS chunks   : {len(scotus_chunks):,}")
print(f"   CUAD chunks     : {len(cuad_chunks):,}")
print(f"   Gold Q&A pairs  : {len(cuad_qa_gold):,}")

print(f"\n✅ Cell 10b complete — ready to continue from Cell 11!")

# CELL 11 — 🔤 BM25 Indexes

In [ ]:
print("🔤 Building BM25 indexes...")
print(f"   SCOTUS chunks : {len(scotus_chunks):,}")
print(f"   CUAD chunks   : {len(cuad_chunks):,}\n")

# ── Tokenizer ─────────────────────────────────────────────────────────────────
def tokenize_bm25(text: str) -> list:
    """
    Legal-aware BM25 tokenizer:
    - Lowercase
    - Keep alphanumeric + hyphens (important for case names)
    - Remove tokens shorter than 2 chars
    - Keep legal abbreviations (U.S., v., et al.)
    """
    text   = text.lower()
    text   = re.sub(r"[^a-z0-9\s\-\.]", " ", text)
    tokens = text.split()
    tokens = [t for t in tokens if len(t) > 2]
    return tokens

# ── Build SCOTUS BM25 ─────────────────────────────────────────────────────────
print("📚 Building SCOTUS BM25 index...")
scotus_bm25_corpus = [
    tokenize_bm25(chunk["text"])
    for chunk in tqdm(scotus_chunks, desc="   Tokenizing SCOTUS")
]
bm25_scotus = BM25Okapi(scotus_bm25_corpus)
print(f"   ✅ SCOTUS BM25 ready — {len(scotus_bm25_corpus):,} documents\n")

# ── Build CUAD BM25 ───────────────────────────────────────────────────────────
print("📜 Building CUAD BM25 index...")
cuad_bm25_corpus = [
    tokenize_bm25(chunk["text"])
    for chunk in tqdm(cuad_chunks, desc="   Tokenizing CUAD  ")
]
bm25_cuad = BM25Okapi(cuad_bm25_corpus)
print(f"   ✅ CUAD BM25 ready — {len(cuad_bm25_corpus):,} documents\n")

# ── Quick test ────────────────────────────────────────────────────────────────
print("🧪 Testing BM25 indexes...")

# Test SCOTUS
test_query   = "probable cause warrant requirement"
test_tokens  = tokenize_bm25(test_query)
scotus_scores = bm25_scotus.get_scores(test_tokens)
top3_scotus  = np.argsort(scotus_scores)[::-1][:3]

print(f"\n   SCOTUS test query : '{test_query}'")
for rank, idx in enumerate(top3_scotus, 1):
    print(f"   [{rank}] score={scotus_scores[idx]:.3f} | {scotus_chunks[idx]['text'][:100]}...")

# Test CUAD
test_query2  = "governing law termination clause"
test_tokens2 = tokenize_bm25(test_query2)
cuad_scores  = bm25_cuad.get_scores(test_tokens2)
top3_cuad    = np.argsort(cuad_scores)[::-1][:3]

print(f"\n   CUAD test query   : '{test_query2}'")
for rank, idx in enumerate(top3_cuad, 1):
    print(f"   [{rank}] score={cuad_scores[idx]:.3f} | {cuad_chunks[idx]['text'][:100]}...")

print(f"\n✅ Cell 11 complete — BM25 indexes ready!")

#CELL 12 — 🧠 Dense Embeddings + FAISS Indexes

In [ ]:
print("🧠 Building dense embeddings + FAISS indexes...")
print(f"   SCOTUS chunks : {len(scotus_chunks):,}")
print(f"   CUAD chunks   : {len(cuad_chunks):,}")
print(f"   ⏳ Expected time: ~55 minutes\n")

# ── Load BGE-M3 embedder ──────────────────────────────────────────────────────
print("📥 Loading BGE-M3 embedder...")
embedder = SentenceTransformer(EMBEDDER_MODEL, device="cuda")
print(f"✅ Embedder ready ({EMBEDDER_MODEL})\n")

# ── Embedding function ────────────────────────────────────────────────────────
def encode_chunks(chunks: list, desc: str) -> np.ndarray:
    """
    Encode list of chunks into normalized float32 embeddings.
    Normalized embeddings → inner product = cosine similarity.
    """
    texts = [c["text"] for c in chunks]
    print(f"   {desc} — {len(texts):,} chunks")
    embs  = embedder.encode(
        texts,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    return embs.astype("float32")

# ── Encode SCOTUS ─────────────────────────────────────────────────────────────
print("📚 Encoding SCOTUS chunks...")
scotus_embeddings = encode_chunks(scotus_chunks, "Encoding SCOTUS")
print(f"   ✅ SCOTUS embeddings : {scotus_embeddings.shape}\n")

# ── Encode CUAD ───────────────────────────────────────────────────────────────
print("📜 Encoding CUAD chunks...")
cuad_embeddings = encode_chunks(cuad_chunks, "Encoding CUAD")
print(f"   ✅ CUAD embeddings   : {cuad_embeddings.shape}\n")

# ── Build FAISS indexes ───────────────────────────────────────────────────────
print("🗂️  Building FAISS indexes...")

faiss_scotus = faiss.IndexFlatIP(EMBEDDING_DIM)
faiss_scotus.add(scotus_embeddings)
print(f"   ✅ SCOTUS FAISS index : {faiss_scotus.ntotal:,} vectors")

faiss_cuad = faiss.IndexFlatIP(EMBEDDING_DIM)
faiss_cuad.add(cuad_embeddings)
print(f"   ✅ CUAD FAISS index   : {faiss_cuad.ntotal:,} vectors\n")

# ── Free GPU memory ───────────────────────────────────────────────────────────
torch.cuda.empty_cache()
print("✅ GPU memory cleared\n")

# ── Quick test ────────────────────────────────────────────────────────────────
print("🧪 Testing FAISS indexes...")

test_query = "probable cause warrant requirement"
query_emb  = embedder.encode(
    [test_query],
    normalize_embeddings=True
).astype("float32")

scores, idx = faiss_scotus.search(query_emb, 3)
print(f"\n   SCOTUS test query : '{test_query}'")
for rank, (i, s) in enumerate(zip(idx[0], scores[0]), 1):
    print(f"   [{rank}] score={s:.4f} | {scotus_chunks[i]['text'][:100]}...")

test_query2  = "governing law termination clause"
query_emb2   = embedder.encode(
    [test_query2],
    normalize_embeddings=True
).astype("float32")

scores2, idx2 = faiss_cuad.search(query_emb2, 3)
print(f"\n   CUAD test query   : '{test_query2}'")
for rank, (i, s) in enumerate(zip(idx2[0], scores2[0]), 1):
    print(f"   [{rank}] score={s:.4f} | {cuad_chunks[i]['text'][:100]}...")

# ── Save indexes to Drive ─────────────────────────────────────────────────────
print("\n💾 Saving FAISS indexes to Drive...")
faiss.write_index(faiss_scotus, f"{DRIVE_SAVE_DIR}/faiss_scotus.bin")
faiss.write_index(faiss_cuad,   f"{DRIVE_SAVE_DIR}/faiss_cuad.bin")

import pickle
with open(f"{DRIVE_SAVE_DIR}/scotus_embeddings.pkl", "wb") as f:
    pickle.dump(scotus_embeddings, f)
with open(f"{DRIVE_SAVE_DIR}/cuad_embeddings.pkl", "wb") as f:
    pickle.dump(cuad_embeddings, f)

print(f"   ✅ faiss_scotus.bin saved")
print(f"   ✅ faiss_cuad.bin saved")
print(f"   ✅ scotus_embeddings.pkl saved")
print(f"   ✅ cuad_embeddings.pkl saved")

print(f"\n✅ Cell 12 complete — FAISS indexes ready!")

# CELL 12b — 📂 Load FAISS Indexes From Drive


In [ ]:
#⚠️ Run this INSTEAD of Cell 12 in future sessions

import faiss
import pickle
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

print("📂 Loading FAISS indexes from Google Drive...\n")

# ── Load FAISS indexes ────────────────────────────────────────────────────────
print("🗂️  Loading FAISS indexes...")
faiss_scotus = faiss.read_index(f"{DRIVE_SAVE_DIR}/faiss_scotus.bin")
faiss_cuad   = faiss.read_index(f"{DRIVE_SAVE_DIR}/faiss_cuad.bin")
print(f"   ✅ faiss_scotus    {faiss_scotus.ntotal:,} vectors")
print(f"   ✅ faiss_cuad      {faiss_cuad.ntotal:,} vectors")

# ── Load embeddings ───────────────────────────────────────────────────────────
print("\n📦 Loading embeddings...")
with open(f"{DRIVE_SAVE_DIR}/scotus_embeddings.pkl", "rb") as f:
    scotus_embeddings = pickle.load(f)
with open(f"{DRIVE_SAVE_DIR}/cuad_embeddings.pkl", "rb") as f:
    cuad_embeddings = pickle.load(f)
print(f"   ✅ scotus_embeddings    {scotus_embeddings.shape}")
print(f"   ✅ cuad_embeddings      {cuad_embeddings.shape}")

# ── Load embedder ─────────────────────────────────────────────────────────────
print(f"\n📥 Loading BGE-M3 embedder...")
embedder = SentenceTransformer(EMBEDDER_MODEL, device="cuda")
print(f"   ✅ Embedder ready ({EMBEDDER_MODEL})")

# ── Quick verification ────────────────────────────────────────────────────────
print(f"\n🧪 Verifying indexes...")
test_query = "probable cause warrant requirement"
query_emb  = embedder.encode(
    [test_query],
    normalize_embeddings=True
).astype("float32")

scores, idx = faiss_scotus.search(query_emb, 1)
print(f"   SCOTUS test score : {scores[0][0]:.4f}")
print(f"   SCOTUS test chunk : {scotus_chunks[idx[0][0]]['text'][:100]}...")

scores2, idx2 = faiss_cuad.search(query_emb, 1)
print(f"   CUAD test score   : {scores2[0][0]:.4f}")

print(f"""
📊 Loaded Summary:
   SCOTUS vectors  : {faiss_scotus.ntotal:,}
   CUAD vectors    : {faiss_cuad.ntotal:,}
   Embedding dim   : {scotus_embeddings.shape[1]}
   Embedder        : {EMBEDDER_MODEL}
""")
print("✅ Cell 12b complete — FAISS indexes ready!")


#CELL 13 — 🔍 Retrieval Functions + RRF Fusion

In [ ]:
print("🔍 Building retrieval functions...\n")

# ── BM25 Tokenizer ────────────────────────────────────────────────────────────
def tokenize_bm25(text: str) -> list:
    """
    Legal-aware BM25 tokenizer.
    Keeps legal abbreviations and case names intact.
    """
    text   = text.lower()
    text   = re.sub(r"[^a-z0-9\s\-\.]", " ", text)
    tokens = text.split()
    tokens = [t for t in tokens if len(t) > 2]
    return tokens

# ── Dense Retrieval ───────────────────────────────────────────────────────────
def dense_retrieve(query: str, corpus: str = "both", k: int = TOPK_DENSE) -> dict:
    """
    Dense retrieval using FAISS.

    Args:
        query  : search query
        corpus : "scotus", "cuad", or "both"
        k      : number of results per corpus

    Returns:
        dict with corpus name → list of (chunk_idx, score)
    """
    query_emb = embedder.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    results = {}

    if corpus in ("scotus", "both"):
        scores, idx      = faiss_scotus.search(query_emb, k)
        results["scotus"] = list(zip(idx[0].tolist(), scores[0].tolist()))

    if corpus in ("cuad", "both"):
        scores, idx     = faiss_cuad.search(query_emb, k)
        results["cuad"] = list(zip(idx[0].tolist(), scores[0].tolist()))

    return results


# ── BM25 Retrieval ────────────────────────────────────────────────────────────
def bm25_retrieve(query: str, corpus: str = "both", k: int = TOPK_BM25) -> dict:
    """
    Sparse retrieval using BM25Okapi.

    Args:
        query  : search query
        corpus : "scotus", "cuad", or "both"
        k      : number of results per corpus

    Returns:
        dict with corpus name → list of (chunk_idx, score)
    """
    tokens  = tokenize_bm25(query)
    results = {}

    if corpus in ("scotus", "both"):
        scores             = bm25_scotus.get_scores(tokens)
        top_idx            = np.argsort(scores)[::-1][:k]
        results["scotus"]  = list(zip(top_idx.tolist(), scores[top_idx].tolist()))

    if corpus in ("cuad", "both"):
        scores            = bm25_cuad.get_scores(tokens)
        top_idx           = np.argsort(scores)[::-1][:k]
        results["cuad"]   = list(zip(top_idx.tolist(), scores[top_idx].tolist()))

    return results


# ── RRF Fusion ────────────────────────────────────────────────────────────────
def rrf_fusion(
    dense_results : dict,
    bm25_results  : dict,
    k             : int = RRF_K,
    topk          : int = TOPK_FUSED
) -> list:
    """
    Reciprocal Rank Fusion (RRF).

    Combines dense and BM25 results using rank positions only.
    Much more robust than weighted score combination.

    Formula: RRF_score = Σ 1 / (k + rank)
    k=60 is the standard damping constant.

    Returns:
        list of (chunk_source, chunk_idx, rrf_score)
        sorted by rrf_score descending
    """
    rrf_scores = {}

    # Process dense results
    for corpus, results in dense_results.items():
        for rank, (chunk_idx, _) in enumerate(results, start=1):
            key                = (corpus, chunk_idx)
            rrf_scores[key]    = rrf_scores.get(key, 0) + 1 / (k + rank)

    # Process BM25 results
    for corpus, results in bm25_results.items():
        for rank, (chunk_idx, _) in enumerate(results, start=1):
            key                = (corpus, chunk_idx)
            rrf_scores[key]    = rrf_scores.get(key, 0) + 1 / (k + rank)

    # Sort by RRF score descending
    sorted_results = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:topk]

    # Return as (corpus, chunk_idx, rrf_score)
    return [(source, idx, score) for (source, idx), score in sorted_results]


# ── Get Chunk Helper ──────────────────────────────────────────────────────────
def get_chunk(source: str, chunk_idx: int) -> dict:
    """
    Get chunk dict by source and index.
    """
    if source == "scotus":
        return scotus_chunks[chunk_idx]
    return cuad_chunks[chunk_idx]


# ── Full Hybrid Retrieval ─────────────────────────────────────────────────────
def hybrid_retrieve(query: str, corpus: str = "both") -> list:
    """
    Full hybrid retrieval pipeline:
    Dense + BM25 → RRF Fusion

    Returns:
        list of (corpus, chunk_idx, rrf_score)
    """
    dense_results = dense_retrieve(query, corpus, TOPK_DENSE)
    bm25_results  = bm25_retrieve(query,  corpus, TOPK_BM25)
    fused         = rrf_fusion(dense_results, bm25_results)
    return fused


# ── Quick Test ────────────────────────────────────────────────────────────────
print("🧪 Testing retrieval functions...\n")

# Test 1: SCOTUS query
query1   = "probable cause warrantless search automobile"
results1 = hybrid_retrieve(query1, corpus="scotus")

print(f"Test 1 — SCOTUS query: '{query1}'")
print(f"   Total fused results : {len(results1)}")
print(f"   Top 3 results:")
for rank, (source, idx, score) in enumerate(results1[:3], 1):
    chunk = get_chunk(source, idx)
    print(f"   [{rank}] rrf={score:.4f} | {source} | {chunk['text'][:100]}...")

# Test 2: CUAD query
query2   = "governing law indemnification clause"
results2 = hybrid_retrieve(query2, corpus="cuad")

print(f"\nTest 2 — CUAD query: '{query2}'")
print(f"   Total fused results : {len(results2)}")
print(f"   Top 3 results:")
for rank, (source, idx, score) in enumerate(results2[:3], 1):
    chunk = get_chunk(source, idx)
    print(f"   [{rank}] rrf={score:.4f} | {source} | {chunk['text'][:100]}...")

# Test 3: Both corpora
query3   = "breach of contract damages"
results3 = hybrid_retrieve(query3, corpus="both")

print(f"\nTest 3 — Both corpora: '{query3}'")
print(f"   Total fused results : {len(results3)}")
print(f"   Corpus distribution:")
scotus_count = sum(1 for r in results3 if r[0] == "scotus")
cuad_count   = sum(1 for r in results3 if r[0] == "cuad")
print(f"   SCOTUS chunks : {scotus_count}")
print(f"   CUAD chunks   : {cuad_count}")
print(f"   Top 3 results:")
for rank, (source, idx, score) in enumerate(results3[:3], 1):
    chunk = get_chunk(source, idx)
    print(f"   [{rank}] rrf={score:.4f} | {source} | {chunk['text'][:100]}...")

print(f"\n✅ Cell 13 complete — retrieval functions ready!")

#Cell 14 — BGE Reranker! 🎯

In [ ]:
print("🏆 Loading BGE reranker...")

# ── Load reranker ─────────────────────────────────────────────────────────────
reranker = CrossEncoder(
    RERANKER_MODEL,
    device="cuda",
    max_length=512
)
print(f"✅ Reranker ready ({RERANKER_MODEL})\n")

# ── Rerank function ───────────────────────────────────────────────────────────
def rerank(query: str, fused_results: list, topk: int = TOPK_RERANK) -> list:
    """
    Rerank fused results using BGE-reranker-large.

    Takes (query, chunk) pairs → scores each pair together
    Much more accurate than vector similarity alone.

    Args:
        query        : search query
        fused_results: list of (corpus, chunk_idx, rrf_score)
        topk         : number of results to return

    Returns:
        list of (corpus, chunk_idx, reranker_score)
        sorted by reranker_score descending
    """
    if not fused_results:
        return []

    # Build (query, chunk_text) pairs
    pairs = []
    for source, chunk_idx, _ in fused_results:
        chunk = get_chunk(source, chunk_idx)
        pairs.append((query, chunk["text"][:1800]))

    # Score all pairs
    scores = reranker.predict(pairs, batch_size=64)

    # Combine with original results
    scored = [
        (source, chunk_idx, float(score))
        for (source, chunk_idx, _), score
        in zip(fused_results, scores)
    ]

    # Sort by reranker score descending
    scored.sort(key=lambda x: x[2], reverse=True)

    return scored[:topk]


# ── Full retrieval pipeline ───────────────────────────────────────────────────
def retrieve(query: str, corpus: str = "both") -> list:
    """
    Complete retrieval pipeline:
    Dense + BM25 → RRF Fusion → Reranking

    Returns:
        list of (corpus, chunk_idx, reranker_score)
    """
    fused   = hybrid_retrieve(query, corpus)
    ranked  = rerank(query, fused, TOPK_RERANK)
    return ranked


# ── Quick test ────────────────────────────────────────────────────────────────
print("🧪 Testing full retrieval pipeline...\n")

# Test 1: SCOTUS
query1   = "probable cause warrantless search automobile"
results1 = retrieve(query1, corpus="scotus")

print(f"Test 1 — SCOTUS: '{query1}'")
for rank, (source, idx, score) in enumerate(results1[:3], 1):
    chunk = get_chunk(source, idx)
    print(f"   [{rank}] rerank={score:.4f} | {chunk['text'][:120]}...")

# Test 2: CUAD
query2   = "governing law indemnification clause"
results2 = retrieve(query2, corpus="cuad")

print(f"\nTest 2 — CUAD: '{query2}'")
for rank, (source, idx, score) in enumerate(results2[:3], 1):
    chunk = get_chunk(source, idx)
    print(f"   [{rank}] rerank={score:.4f} | {chunk['text'][:120]}...")

# Test 3: Score improvement check
print(f"\n📊 Score Improvement (RRF → Reranker):")
print(f"   SCOTUS before rerank : ~0.030 (RRF)")
print(f"   SCOTUS after rerank  : {results1[0][2]:.4f}")
print(f"   CUAD before rerank   : ~0.016 (RRF)")
print(f"   CUAD after rerank    : {results2[0][2]:.4f}")

torch.cuda.empty_cache()

print(f"\n✅ Cell 14 complete — reranker ready!")

# Cell 15 — Query Classifier! 🎯

In [ ]:
print("🎯 Building query classifier...\n")

# ── Intent prototype definitions ──────────────────────────────────────────────
# Each intent has multiple example phrases.
# We embed these → average → get prototype vector.
# Classification = cosine similarity to prototypes.

INTENT_PROTOTYPES = {

    "LEGAL_RAG": [
        "What did the Supreme Court hold in this case?",
        "What is the legal standard for probable cause?",
        "How does the Fourth Amendment apply to searches?",
        "What are Miranda rights in custodial interrogation?",
        "Explain the automobile exception to the warrant requirement",
        "What is qualified immunity for police officers?",
        "How does strict scrutiny apply to fundamental rights?",
        "What is the exclusionary rule in criminal procedure?",
        "Explain the contract termination clause",
        "What are the indemnification provisions in this agreement?",
        "What governing law applies to this contract?",
        "Define the non-compete clause in employment contracts",
        "What is the limitation of liability in this agreement?",
        "Explain breach of contract damages",
        "What is the intellectual property ownership clause?",
    ],

    "LEGAL_SIMPLE": [
        "What is habeas corpus?",
        "Define due process",
        "What does mens rea mean?",
        "What is tort law?",
        "Define promissory estoppel",
        "What is stare decisis?",
        "What does pro bono mean?",
        "Define amicus curiae",
        "What is an injunction?",
        "What does statute of limitations mean?",
    ],

    "GENERAL": [
        "Hello how are you?",
        "What is the weather today?",
        "Tell me a joke",
        "What is artificial intelligence?",
        "How do I cook pasta?",
        "What is the capital of France?",
        "Who won the World Cup?",
        "What is machine learning?",
        "Recommend a good movie",
        "What time is it?",
    ],

    "OUT_OF_SCOPE": [
        "Write me malicious code",
        "How do I hack a system?",
        "Help me commit fraud",
        "How to launder money?",
        "Help me threaten someone",
        "Write fake legal documents",
        "How to evade taxes illegally?",
        "Help me forge a contract",
    ],
}

# ── Build prototype embeddings ────────────────────────────────────────────────
print("📥 Building intent prototype embeddings...")
prototype_embeddings = {}

for intent, phrases in INTENT_PROTOTYPES.items():
    # Embed all phrases for this intent
    embs = embedder.encode(
        phrases,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    # Average → single prototype vector
    prototype = np.mean(embs, axis=0)
    # Renormalize
    prototype = prototype / np.linalg.norm(prototype)
    prototype_embeddings[intent] = prototype
    print(f"   ✅ {intent:<20} ({len(phrases)} examples)")

# ── Classifier function ───────────────────────────────────────────────────────
def classify_query(query: str) -> dict:
    """
    4-way query intent classifier.

    Uses cosine similarity between query embedding
    and intent prototype embeddings.

    Returns:
        dict with:
        - intent    : predicted intent label
        - scores    : similarity scores for all intents
        - confidence: score of top intent
    """
    query_emb = embedder.encode(
        [query],
        normalize_embeddings=True,
        show_progress_bar=False
    )[0]

    # Compute similarity to each prototype
    scores = {}
    for intent, prototype in prototype_embeddings.items():
        scores[intent] = float(np.dot(query_emb, prototype))

    # Get best intent
    best_intent = max(scores, key=scores.get)
    confidence  = scores[best_intent]

    # Apply threshold — if not confident enough → default to LEGAL_RAG
    if confidence < CLASSIFIER_THRESHOLD:
        best_intent = "LEGAL_RAG"

    return {
        "intent"    : best_intent,
        "confidence": round(confidence, 4),
        "scores"    : {k: round(v, 4) for k, v in scores.items()}
    }

# ── Test classifier ───────────────────────────────────────────────────────────
print(f"\n🧪 Testing classifier...\n")

test_queries = [
    # LEGAL_RAG queries
    "What is the automobile exception to the warrant requirement?",
    "Explain probable cause in Fourth Amendment cases",
    "What are the indemnification provisions in this contract?",
    "What governing law applies to this agreement?",
    # LEGAL_SIMPLE queries
    "What is habeas corpus?",
    "Define mens rea",
    # GENERAL queries
    "Hello how are you?",
    "What is the weather today?",
    # OUT_OF_SCOPE queries
    "Help me forge a contract",
    "How do I hack a system?",
]

for query in test_queries:
    result = classify_query(query)
    intent     = result["intent"]
    confidence = result["confidence"]
    emoji = {
        "LEGAL_RAG"    : "⚖️ ",
        "LEGAL_SIMPLE" : "📚",
        "GENERAL"      : "💬",
        "OUT_OF_SCOPE" : "🚫"
    }.get(intent, "❓")
    print(f"   {emoji} [{intent:<15}] {confidence:.3f} | {query[:60]}")

print(f"\n✅ Cell 15 complete — classifier ready!")

#CELL 16 - Router Logic! 🎯

In [ ]:
print("🔀 Building router logic...\n")

# ── Router function ───────────────────────────────────────────────────────────
def route_query(query: str) -> dict:
    """
    Routes query to appropriate handler based on intent.

    LEGAL_RAG    → full agentic RAG pipeline
    LEGAL_SIMPLE → direct LLM (no retrieval)
    GENERAL      → conversational LLM response
    OUT_OF_SCOPE → polite refusal

    Returns:
        dict with intent, action, corpus, response, query
    """
    classification = classify_query(query)
    intent         = classification["intent"]
    confidence     = classification["confidence"]

    # ── OUT_OF_SCOPE ──────────────────────────────────────────────────────────
    if intent == "OUT_OF_SCOPE":
        return {
            "intent"    : intent,
            "action"    : "refuse",
            "confidence": confidence,
            "corpus"    : "N/A",
            "response"  : (
                "I'm a legal research assistant focused on US Supreme Court "
                "case law and contract law. I'm not able to help with that request. "
                "Please ask me about legal cases, constitutional law, or contract analysis."
            ),
            "query"     : query,
        }

    # ── GENERAL ───────────────────────────────────────────────────────────────
    if intent == "GENERAL":
        return {
            "intent"    : intent,
            "action"    : "direct_llm",
            "confidence": confidence,
            "corpus"    : "N/A",
            "response"  : None,   # LLM generates at runtime
            "query"     : query,
        }

    # ── LEGAL_SIMPLE ──────────────────────────────────────────────────────────
    if intent == "LEGAL_SIMPLE":
        return {
            "intent"    : intent,
            "action"    : "direct_llm",
            "confidence": confidence,
            "corpus"    : "N/A",
            "response"  : None,   # LLM generates at runtime
            "query"     : query,
        }

    # ── LEGAL_RAG — Corpus Detection ─────────────────────────────────────────
    # Keywords that strongly indicate CUAD (contract law)
    cuad_keywords = [
        "contract", "agreement", "clause", "provision",
        "indemnif", "terminat", "licens", "intellectual property",
        "confidential", "obligations", "representations",
        "warranties", "governing law", "applicable law",
        "force majeure", "assignment", "sublicense",
    ]

    # Keywords that strongly indicate SCOTUS (case law)
    scotus_keywords = [
        "court", "held", "amendment", "constitutional",
        "statute", "federal", "jurisdiction", "ruling",
        "plaintiff", "defendant", "appeal", "justice",
        "warrant", "probable cause", "fourth amendment",
        "first amendment", "due process", "equal protection",
        "miranda", "habeas", "certiorari", "precedent",
        "automobile exception", "search and seizure",
    ]

    query_lower     = query.lower()
    cuad_score      = sum(1 for kw in cuad_keywords   if kw in query_lower)
    scotus_score    = sum(1 for kw in scotus_keywords  if kw in query_lower)

    if cuad_score > scotus_score:
        corpus = "cuad"
    elif scotus_score > cuad_score:
        corpus = "scotus"
    else:
        corpus = "both"

    return {
        "intent"    : intent,
        "action"    : "rag_pipeline",
        "confidence": confidence,
        "corpus"    : corpus,
        "response"  : None,
        "query"     : query,
    }


# ── Quick LLM call ────────────────────────────────────────────────────────────
def llm_call(
    system_prompt : str,
    user_message  : str,
    max_tokens    : int = 256
) -> str:
    """
    Direct LLM call without retrieval.
    Used for LEGAL_SIMPLE and GENERAL intents.
    Requires generator to be loaded (Cell 24).
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_message},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    response  = tokenizer.decode(generated, skip_special_tokens=True)
    return response.strip()


# ── Test router (routing only — no LLM calls) ─────────────────────────────────
print("🧪 Testing router (routing logic only)...\n")

test_queries = [
    "What is the automobile exception to the warrant requirement?",
    "What are the indemnification provisions in this contract?",
    "What is habeas corpus?",
    "Hello how are you?",
    "Help me forge a contract",
    "Explain the Fourth Amendment search and seizure",
    "What governing law applies to this agreement?",
    "What did the court hold in Miranda v Arizona?",
    "Define the termination clause in this agreement",
]

for query in test_queries:
    result  = route_query(query)
    intent  = result["intent"]
    action  = result["action"]
    corpus  = result.get("corpus", "N/A")
    emoji   = {
        "LEGAL_RAG"    : "⚖️ ",
        "LEGAL_SIMPLE" : "📚",
        "GENERAL"      : "💬",
        "OUT_OF_SCOPE" : "🚫"
    }.get(intent, "❓")
    print(f"   {emoji} [{intent:<15}] → {action:<12} | corpus={corpus:<8} | {query[:55]}")

print(f"\n⚠️  LLM responses activate after Cell 24 loads generator.")
print(f"\n✅ Cell 16 complete — router ready!")

#CELL 17 - 🛠️ Tool Registry

In [ ]:
# CELL 17 — Tool Registry

print("🛠️  Building tool registry...\n")

# ── Tool definitions ──────────────────────────────────────────────────────────
# Each tool is a function the Planner Agent can call.
# Tools are the ACTIONS the agent can take.

def tool_search_scotus(query: str, k: int = TOPK_RERANK) -> dict:
    """
    Tool: Search SCOTUS corpus
    Searches US Supreme Court case law.
    Best for: constitutional law, case holdings,
              legal standards, court opinions.
    """
    results = retrieve(query, corpus="scotus")
    chunks  = [
        {
            "rank"    : i + 1,
            "source"  : source,
            "chunk_id": chunk_idx,
            "score"   : round(score, 4),
            "text"    : get_chunk(source, chunk_idx)["text"][:800],
            "title"   : get_chunk(source, chunk_idx)["title"],
        }
        for i, (source, chunk_idx, score) in enumerate(results[:k])
    ]
    return {
        "tool"      : "search_scotus",
        "query"     : query,
        "corpus"    : "scotus",
        "chunks"    : chunks,
        "top_score" : chunks[0]["score"] if chunks else 0.0,
    }


def tool_search_cuad(query: str, k: int = TOPK_RERANK) -> dict:
    """
    Tool: Search CUAD corpus
    Searches commercial contract database.
    Best for: contract clauses, governing law,
              indemnification, termination, IP rights.
    """
    results = retrieve(query, corpus="cuad")
    chunks  = [
        {
            "rank"    : i + 1,
            "source"  : source,
            "chunk_id": chunk_idx,
            "score"   : round(score, 4),
            "text"    : get_chunk(source, chunk_idx)["text"][:800],
            "title"   : get_chunk(source, chunk_idx)["title"],
        }
        for i, (source, chunk_idx, score) in enumerate(results[:k])
    ]
    return {
        "tool"      : "search_cuad",
        "query"     : query,
        "corpus"    : "cuad",
        "chunks"    : chunks,
        "top_score" : chunks[0]["score"] if chunks else 0.0,
    }


def tool_search_both(query: str, k: int = TOPK_RERANK) -> dict:
    """
    Tool: Search both corpora
    Searches SCOTUS + CUAD simultaneously.
    Best for: queries spanning case law AND contracts,
              unclear corpus, broad legal questions.
    """
    results = retrieve(query, corpus="both")
    chunks  = [
        {
            "rank"    : i + 1,
            "source"  : source,
            "chunk_id": chunk_idx,
            "score"   : round(score, 4),
            "text"    : get_chunk(source, chunk_idx)["text"][:800],
            "title"   : get_chunk(source, chunk_idx)["title"],
        }
        for i, (source, chunk_idx, score) in enumerate(results[:k])
    ]
    return {
        "tool"      : "search_both",
        "query"     : query,
        "corpus"    : "both",
        "chunks"    : chunks,
        "top_score" : chunks[0]["score"] if chunks else 0.0,
    }


def tool_rewrite_query(original_query: str, feedback: str) -> dict:
    """
    Tool: Rewrite query
    Reformulates a failed query based on feedback.
    Best for: when retrieval scores are too low,
              when chunks are irrelevant,
              when query is too broad or too narrow.
    """
    # Rule-based rewriting strategies
    strategies = []

    # Strategy 1: Add legal context
    if "what is" in original_query.lower():
        rewritten = original_query.replace(
            "what is", "define legal concept"
        )
        strategies.append(rewritten)

    # Strategy 2: Add domain keywords
    if "contract" not in original_query.lower() and "court" not in original_query.lower():
        strategies.append(f"legal {original_query}")

    # Strategy 3: Simplify to core terms
    words    = original_query.lower().split()
    keywords = [w for w in words if len(w) > 4]
    if keywords:
        strategies.append(" ".join(keywords[:5]))

    # Strategy 4: Expand abbreviations
    expansions = {
        "4th amendment" : "Fourth Amendment",
        "1st amendment" : "First Amendment",
        "k"             : "contract",
        "ip"            : "intellectual property",
    }
    rewritten = original_query
    for abbr, full in expansions.items():
        rewritten = rewritten.replace(abbr, full)
    if rewritten != original_query:
        strategies.append(rewritten)

    # Pick best strategy
    best_rewrite = strategies[0] if strategies else f"{original_query} legal definition"

    return {
        "tool"           : "rewrite_query",
        "original_query" : original_query,
        "rewritten_query": best_rewrite,
        "feedback"       : feedback,
        "strategies"     : strategies,
    }


def tool_summarize_context(chunks: list, max_chars: int = 3000) -> dict:
    """
    Tool: Summarize context
    Compresses retrieved chunks when context is too long.
    Best for: when top-k chunks exceed token budget,
              when many chunks have overlapping content.
    """
    if not chunks:
        return {"tool": "summarize_context", "summary": "", "chunks_used": 0}

    # Simple extractive summarization
    # Take first sentence of each chunk
    summaries = []
    total     = 0

    for chunk in chunks:
        text  = chunk.get("text", "")
        # Get first 2 sentences
        sents = text.split(". ")[:2]
        brief = ". ".join(sents) + "."
        if total + len(brief) < max_chars:
            summaries.append(f"[S{chunk['rank']}] {brief}")
            total += len(brief)

    summary = "\n\n".join(summaries)

    return {
        "tool"       : "summarize_context",
        "summary"    : summary,
        "chunks_used": len(summaries),
        "total_chars": total,
    }


# ── Tool Registry ─────────────────────────────────────────────────────────────
TOOL_REGISTRY = {
    "search_scotus"    : {
        "function"   : tool_search_scotus,
        "description": "Search US Supreme Court case law (SCOTUS corpus)",
        "best_for"   : "constitutional law, case holdings, legal standards",
        "corpus"     : "scotus",
    },
    "search_cuad"      : {
        "function"   : tool_search_cuad,
        "description": "Search commercial contracts (CUAD corpus)",
        "best_for"   : "contract clauses, governing law, indemnification",
        "corpus"     : "cuad",
    },
    "search_both"      : {
        "function"   : tool_search_both,
        "description": "Search both SCOTUS and CUAD corpora",
        "best_for"   : "broad legal questions, unclear corpus",
        "corpus"     : "both",
    },
    "rewrite_query"    : {
        "function"   : tool_rewrite_query,
        "description": "Rewrite a failed query to improve retrieval",
        "best_for"   : "low retrieval scores, irrelevant results",
        "corpus"     : "N/A",
    },
    "summarize_context": {
        "function"   : tool_summarize_context,
        "description": "Compress retrieved chunks to fit token budget",
        "best_for"   : "too many chunks, overlapping content",
        "corpus"     : "N/A",
    },
}

# ── Tool caller ───────────────────────────────────────────────────────────────
def call_tool(tool_name: str, **kwargs) -> dict:
    """
    Unified tool caller.
    Planner Agent calls this with tool name + arguments.
    """
    if tool_name not in TOOL_REGISTRY:
        return {
            "error"    : f"Tool '{tool_name}' not found",
            "available": list(TOOL_REGISTRY.keys())
        }
    return TOOL_REGISTRY[tool_name]["function"](**kwargs)


# ── Test tools ────────────────────────────────────────────────────────────────
print("🧪 Testing all tools...\n")

# Test search_scotus
print("1️⃣  search_scotus:")
r1 = call_tool("search_scotus", query="probable cause automobile exception")
print(f"   top_score : {r1['top_score']}")
print(f"   chunks    : {len(r1['chunks'])}")
print(f"   preview   : {r1['chunks'][0]['text'][:100]}...\n")

# Test search_cuad
print("2️⃣  search_cuad:")
r2 = call_tool("search_cuad", query="termination clause notice period")
print(f"   top_score : {r2['top_score']}")
print(f"   chunks    : {len(r2['chunks'])}")
print(f"   preview   : {r2['chunks'][0]['text'][:100]}...\n")

# Test search_both
print("3️⃣  search_both:")
r3 = call_tool("search_both", query="breach of contract damages")
print(f"   top_score : {r3['top_score']}")
print(f"   chunks    : {len(r3['chunks'])}")
scotus_n = sum(1 for c in r3['chunks'] if c['source'] == 'scotus')
cuad_n   = sum(1 for c in r3['chunks'] if c['source'] == 'cuad')
print(f"   scotus    : {scotus_n} | cuad: {cuad_n}\n")

# Test rewrite_query
print("4️⃣  rewrite_query:")
r4 = call_tool(
    "rewrite_query",
    original_query="ip clause",
    feedback="low retrieval score"
)
print(f"   original  : {r4['original_query']}")
print(f"   rewritten : {r4['rewritten_query']}\n")

# Test summarize_context
print("5️⃣  summarize_context:")
r5 = call_tool("summarize_context", chunks=r1['chunks'][:3])
print(f"   chunks_used : {r5['chunks_used']}")
print(f"   total_chars : {r5['total_chars']}")
print(f"   preview     : {r5['summary'][:150]}...\n")

print("📋 Registered tools:")
for name, info in TOOL_REGISTRY.items():
    print(f"   ✅ {name:<20} → {info['description']}")

print(f"\n✅ Cell 17 complete — tool registry ready!")

# CELL 18 — 🧠 LangGraph Agent State Definition

In [ ]:
from typing import TypedDict, Annotated, List, Optional, Literal
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
import operator

print("🧠 Defining LangGraph agent state...\n")

# ── Agent State ───────────────────────────────────────────────────────────────
# This TypedDict defines EVERYTHING that flows between agents.
# Every node reads from this state and writes back to it.
# Think of it as a shared whiteboard all agents can see.

class AgentState(TypedDict):

    # ── Query Info ────────────────────────────────────────────────────────────
    query           : str                  # original user query
    intent          : str                  # LEGAL_RAG / LEGAL_SIMPLE / GENERAL / OUT_OF_SCOPE
    corpus          : str                  # scotus / cuad / both
    confidence      : float                # classifier confidence score

    # ── Agent Loop Control ────────────────────────────────────────────────────
    hop_count       : int                  # current retrieval hop (max 3)
    rewrite_count   : int                  # current rewrite attempt (max 2)
    regen_count     : int                  # current regeneration attempt (max 1)

    # ── Retrieval State ───────────────────────────────────────────────────────
    current_query   : str                  # possibly rewritten query
    retrieved_chunks: List[dict]           # accumulated chunks across hops
    retrieval_scores: List[float]          # reranker scores for chunks
    top_score       : float                # best reranker score so far

    # ── Reflection State ──────────────────────────────────────────────────────
    reflection      : str                  # CORRECT / AMBIGUOUS / INCORRECT
    reflection_feedback : str             # why reflection failed (for rewriter)

    # ── Generation State ──────────────────────────────────────────────────────
    context         : str                  # formatted context for LLM
    answer          : str                  # generated answer
    thinking        : str                  # CoT reasoning chain

    # ── Verification State ────────────────────────────────────────────────────
    verified        : bool                 # did verification pass?
    verification_feedback : str           # why verification failed

    # ── Final Output ──────────────────────────────────────────────────────────
    final_answer    : str                  # clean final answer
    sources         : List[dict]           # formatted source citations
    pipeline_status : str                  # SUCCESS / FAILED / REFUSED

    # ── Reasoning Trace ───────────────────────────────────────────────────────
    # Full log of every agent decision — great for thesis diagrams!
    reasoning_trace : Annotated[List[str], operator.add]


# ── State initializer ─────────────────────────────────────────────────────────
def init_state(query: str) -> AgentState:
    """
    Initialize a fresh agent state for a new query.
    All counters start at 0, all lists empty.
    """
    return AgentState(
        # Query info
        query                = query,
        intent               = "",
        corpus               = "both",
        confidence           = 0.0,

        # Loop control
        hop_count            = 0,
        rewrite_count        = 0,
        regen_count          = 0,

        # Retrieval
        current_query        = query,
        retrieved_chunks     = [],
        retrieval_scores     = [],
        top_score            = 0.0,

        # Reflection
        reflection           = "",
        reflection_feedback  = "",

        # Generation
        context              = "",
        answer               = "",
        thinking             = "",

        # Verification
        verified             = False,
        verification_feedback= "",

        # Output
        final_answer         = "",
        sources              = [],
        pipeline_status      = "RUNNING",

        # Trace
        reasoning_trace      = [],
    )


# ── Test state ────────────────────────────────────────────────────────────────
print("🧪 Testing state initialization...\n")

test_state = init_state("What is the automobile exception?")

print("📋 Initial state:")
print(f"   query           : {test_state['query']}")
print(f"   intent          : '{test_state['intent']}' (empty until classifier runs)")
print(f"   corpus          : {test_state['corpus']}")
print(f"   hop_count       : {test_state['hop_count']}")
print(f"   rewrite_count   : {test_state['rewrite_count']}")
print(f"   retrieved_chunks: {test_state['retrieved_chunks']} (empty until retrieval)")
print(f"   top_score       : {test_state['top_score']}")
print(f"   pipeline_status : {test_state['pipeline_status']}")
print(f"   reasoning_trace : {test_state['reasoning_trace']} (empty until agents run)")

print(f"\n📋 State fields total: {len(AgentState.__annotations__)}")
print(f"   Query fields     : 4")
print(f"   Control fields   : 3")
print(f"   Retrieval fields : 5")
print(f"   Reflection fields: 2")
print(f"   Generation fields: 3")
print(f"   Verification fields: 2")
print(f"   Output fields    : 3")
print(f"   Trace fields     : 1")

print(f"\n✅ Cell 18 complete — agent state ready!")

# CELL 19 — 🎯 Planner Agent (ReAct)

In [ ]:
print("🎯 Building Planner Agent (ReAct)...\n")

def planner_agent(state: AgentState) -> AgentState:
    """
    Planner Agent — ReAct Pattern

    Reasons about the query and decides:
    1. Which corpus to search (scotus/cuad/both)
    2. Which tool to use
    3. How to formulate the search query

    Updates state with:
    - intent
    - corpus
    - current_query
    - reasoning_trace entry
    """

    query = state["query"]

    # ── Step 1: Classify intent ───────────────────────────────────────────────
    route  = route_query(query)
    intent = route["intent"]
    corpus = route.get("corpus", "both")

    # ── Step 2: ReAct Reasoning ───────────────────────────────────────────────
    # Thought process logged to reasoning_trace
    thought = f"[PLANNER] Query: '{query}'\n"
    thought += f"  → Intent    : {intent}\n"
    thought += f"  → Corpus    : {corpus}\n"
    thought += f"  → Confidence: {route['confidence']}\n"

    # ── Step 3: Query optimization ────────────────────────────────────────────
    # Expand short queries for better retrieval
    current_query = query
    if len(query.split()) < 4:
        # Too short → add legal context
        current_query = f"legal {query} definition application"
        thought += f"  → Query expanded (too short): '{current_query}'\n"
    else:
        thought += f"  → Query OK — no expansion needed\n"

    # ── Step 4: Handle non-RAG intents immediately ────────────────────────────
    if intent == "OUT_OF_SCOPE":
        thought += f"  → Action: REFUSE\n"
        return {
            **state,
            "intent"         : intent,
            "corpus"         : corpus,
            "confidence"     : route["confidence"],
            "current_query"  : current_query,
            "final_answer"   : route["response"],
            "pipeline_status": "REFUSED",
            "reasoning_trace": [thought],
        }

    if intent in ("GENERAL", "LEGAL_SIMPLE"):
        thought += f"  → Action: DIRECT_LLM (no retrieval needed)\n"
        return {
            **state,
            "intent"         : intent,
            "corpus"         : corpus,
            "confidence"     : route["confidence"],
            "current_query"  : current_query,
            "pipeline_status": "DIRECT_LLM",
            "reasoning_trace": [thought],
        }

    # ── Step 5: Plan retrieval for LEGAL_RAG ─────────────────────────────────
    thought += f"  → Action: RAG_PIPELINE\n"
    thought += f"  → Tool selected: search_{corpus}\n"
    thought += f"  → Hop: {state['hop_count'] + 1}/{MAX_HOPS}\n"

    return {
        **state,
        "intent"         : intent,
        "corpus"         : corpus,
        "confidence"     : route["confidence"],
        "current_query"  : current_query,
        "pipeline_status": "RUNNING",
        "reasoning_trace": [thought],
    }


# ── Test planner ──────────────────────────────────────────────────────────────
print("🧪 Testing Planner Agent...\n")

test_cases = [
    "What is the automobile exception to the warrant requirement?",
    "What are the indemnification provisions in this contract?",
    "What is habeas corpus?",
    "Hello how are you today?",
    "Help me forge a legal document",
    "ip",   # short query test
]

for query in test_cases:
    state  = init_state(query)
    result = planner_agent(state)

    print(f"Query  : {query[:60]}")
    print(f"Intent : {result['intent']}")
    print(f"Corpus : {result['corpus']}")
    print(f"Status : {result['pipeline_status']}")
    print(f"Trace  :\n{result['reasoning_trace'][0]}")
    print("─" * 50)


#CELL 20 — 🔎 Retrieval Agent

In [ ]:
print("🔎 Building Retrieval Agent...\n")

def retrieval_agent(state: AgentState) -> AgentState:
    """
    Retrieval Agent
    ✅ FIX: deduplication now uses (source, title, text[:50])
    instead of (source, chunk_id) which was comparing rank
    positions across hops, not actual chunk identity.
    """
    query  = state["current_query"]
    corpus = state["corpus"]
    hop    = state["hop_count"] + 1

    thought  = f"[RETRIEVAL] Hop {hop}/{MAX_HOPS}\n"
    thought += f"  → Query  : '{query}'\n"
    thought += f"  → Corpus : {corpus}\n"

    tool_name = f"search_{corpus}"
    results   = call_tool(tool_name, query=query)

    if "error" in results:
        thought += f"  → Tool error: {results['error']}\n"
        thought += f"  → Falling back to search_both\n"
        results  = call_tool("search_both", query=query)

    new_chunks = results.get("chunks", [])
    top_score  = results.get("top_score", 0.0)

    thought += f"  → Tool     : {tool_name}\n"
    thought += f"  → Retrieved: {len(new_chunks)} chunks\n"
    thought += f"  → Top score: {top_score:.4f}\n"

    # ✅ FIX: use actual chunk identity for dedup, not rank position
    existing_ids = {
        (c["source"], c["title"], c["text"][:50])
        for c in state["retrieved_chunks"]
    }

    unique_chunks = [
        c for c in new_chunks
        if (c["source"], c["title"], c["text"][:50]) not in existing_ids
    ]

    thought += f"  → Unique   : {len(unique_chunks)} new chunks\n"

    all_chunks = state["retrieved_chunks"] + unique_chunks
    all_scores = [c["score"] for c in all_chunks]
    best_score = max(all_scores) if all_scores else 0.0

    thought += f"  → Total accumulated: {len(all_chunks)} chunks\n"
    thought += f"  → Best score so far: {best_score:.4f}\n"

    return {
        **state,
        "retrieved_chunks": all_chunks,
        "retrieval_scores": all_scores,
        "top_score"       : best_score,
        "hop_count"       : hop,
        "reasoning_trace" : [thought],
    }


print("🧪 Testing Retrieval Agent...\n")

query = "What is the automobile exception to the warrant requirement?"
state = init_state(query)
state = planner_agent(state)
state = retrieval_agent(state)

print(f"Query           : {query[:60]}")
print(f"Corpus searched : {state['corpus']}")
print(f"Hop count       : {state['hop_count']}")
print(f"Chunks retrieved: {len(state['retrieved_chunks'])}")
print(f"Top score       : {state['top_score']:.4f}")
print(f"\nTop 3 chunks:")
for chunk in state["retrieved_chunks"][:3]:
    print(f"   [{chunk['rank']}] score={chunk['score']:.4f} | {chunk['text'][:100]}...")

print(f"\nReasoning trace:")
for trace in state["reasoning_trace"]:
    print(trace)

print("✅ Cell 20 complete — Retrieval Agent ready!")

#Cell 21 — Reflection Agent! 🎯

In [ ]:
# CELL 21 — Reflection Agent (CRAG-style)

print("🪞 Building Reflection Agent...\n")

def reflection_agent(state: AgentState) -> AgentState:
    """
    Reflection Agent — CRAG Style

    Evaluates quality of retrieved chunks.
    Decides whether to:
    → CORRECT   : proceed to generation
    → AMBIGUOUS : proceed with caution
    → INCORRECT : trigger query rewriter

    Updates state with:
    - reflection (CORRECT/AMBIGUOUS/INCORRECT)
    - reflection_feedback
    - reasoning_trace entry
    """

    chunks    = state["retrieved_chunks"]
    top_score = state["top_score"]
    query     = state["current_query"]
    thought   = f"[REFLECTION] Evaluating retrieval quality\n"
    thought  += f"  → Query     : '{query}'\n"
    thought  += f"  → Chunks    : {len(chunks)}\n"
    thought  += f"  → Top score : {top_score:.4f}\n"

    # ── Score-based classification ────────────────────────────────────────────
    if top_score >= CORRECT_THRESHOLD:
        reflection = "CORRECT"
        feedback   = "Retrieval quality is excellent — proceed to generation."
        thought   += f"  → Score {top_score:.4f} >= {CORRECT_THRESHOLD} → CORRECT ✅\n"

    elif top_score >= AMBIGUOUS_LOW:
        reflection = "AMBIGUOUS"
        feedback   = (
            f"Retrieval quality is acceptable but not optimal "
            f"(score={top_score:.4f}). Proceeding with caution."
        )
        thought += f"  → Score {top_score:.4f} in [{AMBIGUOUS_LOW}, {CORRECT_THRESHOLD}] → AMBIGUOUS 🟡\n"

    else:
        reflection = "INCORRECT"
        feedback   = (
            f"Retrieval quality is poor (score={top_score:.4f}). "
            f"Query needs rewriting. "
            f"Original query may be too broad or use wrong terminology."
        )
        thought += f"  → Score {top_score:.4f} < {AMBIGUOUS_LOW} → INCORRECT 🔴\n"

    # ── Additional checks ─────────────────────────────────────────────────────
    # Check if chunks are actually relevant (content check)
    if chunks and reflection != "INCORRECT":
        # Check if top chunk has enough content
        top_chunk_len = len(chunks[0].get("text", ""))
        if top_chunk_len < 100:
            reflection = "INCORRECT"
            feedback   = "Top chunk too short — likely not useful."
            thought   += f"  → Top chunk too short ({top_chunk_len} chars) → INCORRECT 🔴\n"

    # Check if we have enough chunks
    if len(chunks) < 3 and reflection == "CORRECT":
        reflection = "AMBIGUOUS"
        feedback   = "Too few chunks retrieved — proceeding with caution."
        thought   += f"  → Only {len(chunks)} chunks — downgraded to AMBIGUOUS 🟡\n"

    # ── Max hops check ────────────────────────────────────────────────────────
    # If we've hit max hops → accept whatever we have
    if state["hop_count"] >= MAX_HOPS and reflection == "INCORRECT":
        reflection = "AMBIGUOUS"
        feedback   = f"Max hops ({MAX_HOPS}) reached — proceeding with best available."
        thought   += f"  → Max hops reached → forced AMBIGUOUS\n"

    thought += f"  → Decision : {reflection}\n"
    thought += f"  → Feedback : {feedback}\n"

    return {
        **state,
        "reflection"        : reflection,
        "reflection_feedback": feedback,
        "reasoning_trace"   : [thought],
    }


# ── Test reflection agent ─────────────────────────────────────────────────────
print("🧪 Testing Reflection Agent...\n")

# Test 1: Good retrieval (should be CORRECT or AMBIGUOUS)
query  = "What is the automobile exception to the warrant requirement?"
state  = init_state(query)
state  = planner_agent(state)
state  = retrieval_agent(state)
state  = reflection_agent(state)

print(f"Test 1 — Good query:")
print(f"   Top score  : {state['top_score']:.4f}")
print(f"   Reflection : {state['reflection']}")
print(f"   Feedback   : {state['reflection_feedback']}")
print(f"   Trace      :\n{state['reasoning_trace'][-1]}")

# Test 2: Bad retrieval (should be INCORRECT)
# Simulate low score
bad_state = init_state("xyzzy legal gobbledygook nonsense")
bad_state = planner_agent(bad_state)
bad_state = retrieval_agent(bad_state)
bad_state = reflection_agent(bad_state)

print(f"\nTest 2 — Bad query:")
print(f"   Top score  : {bad_state['top_score']:.4f}")
print(f"   Reflection : {bad_state['reflection']}")
print(f"   Feedback   : {bad_state['reflection_feedback']}")

print(f"\n✅ Cell 21 complete — Reflection Agent ready!")

#CELL 22 — ✏️ Query Rewriter Agent

In [ ]:

print("✏️  Building Query Rewriter Agent...\n")

def clean_query(text: str) -> str:
    """
    Remove question words and stopwords from query.
    Returns clean keyword string.
    """
    # Remove question words
    question_words = [
        "what is the", "what are the", "what is a",
        "what are", "what is", "what does", "what did",
        "how does the", "how do the", "how does",
        "how do", "how is", "how are",
        "explain the", "explain how", "explain",
        "describe the", "describe",
        "tell me about", "can you explain",
        "define the", "define",
        "what", "how", "why", "when", "where", "who",
    ]
    cleaned = text.lower().strip("?").strip()
    # Sort by length descending to match longer phrases first
    for qw in sorted(question_words, key=len, reverse=True):
        if cleaned.startswith(qw):
            cleaned = cleaned[len(qw):].strip()
            break

    # Remove common stopwords but keep legal terms
    stopwords = {"the", "a", "an", "is", "are", "was",
                 "in", "on", "at", "to", "for", "of",
                 "and", "or", "but", "this", "that"}

    words   = cleaned.split()
    cleaned = " ".join(w for w in words if w not in stopwords)
    return cleaned.strip()


def rewriter_agent(state: AgentState) -> AgentState:
    """
    Query Rewriter Agent

    Rewrites the current query when reflection
    says retrieval quality is INCORRECT.

    Strategies:
    1. Remove question words → add domain prefix
    2. Extract key legal terms only + expand corpus

    Updates state with:
    - current_query (rewritten)
    - rewrite_count
    - corpus (possibly expanded)
    - reasoning_trace entry
    """

    original_query = state["current_query"]
    feedback       = state["reflection_feedback"]
    rewrite_count  = state["rewrite_count"] + 1
    corpus         = state["corpus"]

    thought  = f"[REWRITER] Attempt {rewrite_count}/{MAX_REWRITES}\n"
    thought += f"  → Original : '{original_query}'\n"
    thought += f"  → Feedback : {feedback}\n"

    # ── Clean query ───────────────────────────────────────────────────────────
    cleaned = clean_query(original_query)
    thought += f"  → Cleaned  : '{cleaned}'\n"

    # ── Strategy 1 (first rewrite) ────────────────────────────────────────────
    if rewrite_count == 1:
        # Add domain prefix based on corpus
        domain_prefix = {
            "scotus": "supreme court",
            "cuad"  : "contract clause",
            "both"  : "legal",
        }
        prefix    = domain_prefix.get(corpus, "legal")
        rewritten = f"{prefix} {cleaned}"
        strategy  = f"domain prefix '{prefix}' + cleaned query"

    # ── Strategy 2 (second rewrite) ───────────────────────────────────────────
    else:
        # Extract legal keywords only
        legal_terms = [
            "amendment", "warrant", "search", "seizure",
            "probable cause", "reasonable", "constitutional",
            "contract", "agreement", "clause", "provision",
            "indemnif", "terminat", "govern", "liabilit",
            "exception", "standard", "test", "rule", "right",
            "court", "held", "judgment", "damages", "breach",
            "exclusionary", "miranda", "fourth", "first",
            "fifth", "sixth", "due process", "equal protection",
        ]
        words     = cleaned.split()
        key_terms = [
            w for w in words
            if any(lt in w.lower() for lt in legal_terms)
        ]
        # Keep at least 3 words even if not legal terms
        if len(key_terms) < 3:
            key_terms = words[:5]

        rewritten  = " ".join(key_terms)
        strategy   = "extract legal keywords"

        # Expand corpus on second rewrite
        corpus     = "both"
        thought   += f"  → Expanding corpus to 'both'\n"

    thought += f"  → Strategy : {strategy}\n"
    thought += f"  → Rewritten: '{rewritten}'\n"
    thought += f"  → Corpus   : {corpus}\n"

    return {
        **state,
        "current_query"  : rewritten,
        "corpus"         : corpus,
        "rewrite_count"  : rewrite_count,
        "reasoning_trace": [thought],
    }


# ── Test rewriter agent ───────────────────────────────────────────────────────
print("🧪 Testing Rewriter Agent...\n")

test_queries = [
    "What is the exclusionary rule?",
    "How does the automobile exception apply?",
    "Explain the indemnification clause in contracts",
    "What are Miranda rights?",
]

for query in test_queries:
    state = init_state(query)
    state = planner_agent(state)

    # First rewrite
    state["reflection"]          = "INCORRECT"
    state["reflection_feedback"] = "Low retrieval score"
    state = rewriter_agent(state)

    rewrite1 = state["current_query"]
    corpus1  = state["corpus"]

    # Second rewrite
    state["reflection"]          = "INCORRECT"
    state["reflection_feedback"] = "Still low score"
    state = rewriter_agent(state)

    rewrite2 = state["current_query"]
    corpus2  = state["corpus"]

    print(f"Original  : '{query}'")
    print(f"Rewrite 1 : '{rewrite1}' | corpus={corpus1}")
    print(f"Rewrite 2 : '{rewrite2}' | corpus={corpus2}")
    print("─" * 55)

print(f"\n✅ Cell 22 complete — Rewriter Agent ready!")

#CELL 23 — ⚡ Generation Agent (Qwen2.5-32B + CoT)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("⚡ Building Generation Agent...\n")

print(f"📥 Loading {GEN_MODEL}...")
print(f"   ⏳ ~4 minutes...\n")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(
    GEN_MODEL,
    use_fast=True
)

model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print(f"✅ {GEN_MODEL} loaded!\n")

# ✅ FIX: answer length cap raised — contract clause answers
# need more sentences to match gold answers properly
LEGAL_SYSTEM_PROMPT = """You are a precise US legal research assistant.
You have access to US Supreme Court opinions and commercial contracts.

STRICT RULES:
1. Answer using ONLY the provided SOURCES below
2. NEVER use outside knowledge — only what is in SOURCES
3. Every claim MUST have a citation [S1], [S2], etc.
4. If answer is not in SOURCES → say exactly: "Not found in provided sources."
5. For case law questions: answer in 3-8 sentences
6. For contract clause questions: answer in up to 12 sentences
7. End your answer with <END>

THINKING FORMAT:
First think inside <think> tags:
- What does each source say?
- Which sources are most relevant?
- What is the direct answer?

Then give your grounded answer with citations."""


def format_context(chunks: list, max_chars: int = 4500) -> str:
    """
    Format retrieved chunks into numbered source block.
    ✅ FIX: shortened headers from ~70 chars to ~5 chars
    so more actual legal text fits in the context budget.
    """
    parts  = []
    total  = 0
    for i, chunk in enumerate(chunks[:10], 1):
        text   = chunk.get("text",  "")[:600]
        # ✅ FIX: was f"[S{i}] ({source.upper()}) {title}" — wasted
        # ~70 chars per chunk on metadata, ~700 chars total for 10 chunks
        entry  = f"[S{i}]\n{text}"
        if total + len(entry) > max_chars:
            break
        parts.append(entry)
        total += len(entry)
    return "\n\n".join(parts)


def generate_answer(
    query   : str,
    context : str,
    system  : str = LEGAL_SYSTEM_PROMPT,
    max_new : int = MAX_NEW_TOKENS
) -> tuple:
    """
    Generate grounded answer with CoT reasoning.
    ✅ FIX: switched from greedy (do_sample=False) to
    low-temperature sampling — greedy decoding produces
    repetitive hedging answers that score lower on
    Answer Relevancy and Faithfulness.
    """
    user_message = f"QUESTION:\n{query}\n\nSOURCES:\n{context}\n\nProvide your answer:"

    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user_message},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new,
            # ✅ FIX: was do_sample=False, temperature=1.0 (contradictory)
            # low-temperature sampling produces more precise, varied answers
            do_sample=True,
            temperature=0.1,
            top_p=0.9,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    full_text  = tokenizer.decode(generated, skip_special_tokens=True).strip()

    thinking = ""
    answer   = full_text

    if "<think>" in full_text and "</think>" in full_text:
        think_start = full_text.find("<think>") + len("<think>")
        think_end   = full_text.find("</think>")
        thinking    = full_text[think_start:think_end].strip()
        answer      = full_text[think_end + len("</think>"):].strip()

    if "<END>" in answer:
        answer = answer.split("<END>")[0].strip()

    return thinking, answer


def generation_agent(state: AgentState) -> AgentState:
    query   = state["query"]
    chunks  = state["retrieved_chunks"]
    thought = f"[GENERATION] Generating answer\n"
    thought += f"  → Query   : '{query}'\n"
    thought += f"  → Chunks  : {len(chunks)}\n"

    context  = format_context(chunks)
    thought += f"  → Context : {len(context)} chars\n"

    thinking, answer = generate_answer(query, context)

    thought += f"  → Thinking: {len(thinking)} chars\n"
    thought += f"  → Answer  : {len(answer)} chars\n"

    return {
        **state,
        "context"        : context,
        "thinking"       : thinking,
        "answer"         : answer,
        "reasoning_trace": [thought],
    }


print("🧪 Testing Generation Agent...\n")

query = "What is the automobile exception to the warrant requirement?"
state = init_state(query)
state = planner_agent(state)
state = retrieval_agent(state)
state = reflection_agent(state)
state = generation_agent(state)

print(f"Query   : {query}")
print(f"Context : {len(state['context'])} chars")
print(f"\n🧠 Thinking:")
print(f"{state['thinking'][:400]}..." if state['thinking'] else "   (no thinking block)")
print(f"\n💬 Answer:")
print(f"{state['answer']}")
print(f"\n✅ Cell 23 complete — Generation Agent ready!")

# CELL 24 — Verification Agent


In [ ]:
print("✅ Building Verification Agent...\n")

def verification_agent(state: AgentState) -> AgentState:
    """
    Verification Agent
    ✅ FIX: relevance threshold raised from 0.10 → 0.15
    so the agent actually catches low-quality answers
    instead of letting everything pass through.
    """
    query   = state["query"]
    answer  = state["answer"]
    chunks  = state["retrieved_chunks"]
    regen   = state["regen_count"]
    thought = f"[VERIFICATION] Checking answer quality\n"
    thought += f"  → Query    : '{query[:60]}'\n"
    thought += f"  → Answer   : {len(answer)} chars\n"

    # ── Check 1: Answer exists ────────────────────────────────────────────────
    if not answer or len(answer.strip()) < 20:
        thought += f"  → FAIL: Answer too short or empty\n"
        return {
            **state,
            "verified"             : False,
            "verification_feedback": "Answer is empty or too short.",
            "pipeline_status"      : "FAILED",
            "reasoning_trace"      : [thought],
        }

    # ── Check 2: Not found response ───────────────────────────────────────────
    if "not found in provided sources" in answer.lower():
        thought += f"  → INFO: Model says not found in sources\n"
        return {
            **state,
            "verified"             : True,
            "verification_feedback": "Model correctly identified missing information.",
            "final_answer"         : answer,
            "sources"              : [],
            "pipeline_status"      : "SUCCESS",
            "reasoning_trace"      : [thought],
        }

    # ── Check 3: Has citations ────────────────────────────────────────────────
    import re
    citations = re.findall(r"\[S\d+\]", answer)
    if not citations:
        thought += f"  → WARN: No citations found in answer\n"
        if regen < MAX_REGENERATIONS:
            thought += f"  → Triggering regeneration\n"
            return {
                **state,
                "verified"             : False,
                "verification_feedback": "Answer has no citations — regenerating.",
                "regen_count"          : regen + 1,
                "pipeline_status"      : "REGENERATE",
                "reasoning_trace"      : [thought],
            }
    else:
        thought += f"  → Citations found: {citations}\n"

    # ── Check 4: Faithfulness check ───────────────────────────────────────────
    max_source_num    = len(chunks)
    valid_citations   = []
    invalid_citations = []

    for citation in citations:
        num = int(re.search(r"\d+", citation).group())
        if num <= max_source_num:
            valid_citations.append(citation)
        else:
            invalid_citations.append(citation)

    if invalid_citations:
        thought += f"  → WARN: Invalid citations: {invalid_citations}\n"
    else:
        thought += f"  → All citations valid ✅\n"

    # ── Check 5: Answer relevance ─────────────────────────────────────────────
    query_words  = set(query.lower().split())
    answer_words = set(answer.lower().split())
    overlap      = query_words & answer_words
    relevance    = len(overlap) / max(len(query_words), 1)

    thought += f"  → Relevance score: {relevance:.2f}\n"

    # ✅ FIX: raised from 0.10 → 0.15 so verification actually
    # catches irrelevant answers instead of being a no-op
    if relevance < 0.15 and regen < MAX_REGENERATIONS:
        thought += f"  → FAIL: Answer not relevant to query\n"
        return {
            **state,
            "verified"             : False,
            "verification_feedback": f"Answer not relevant (overlap={relevance:.2f})",
            "regen_count"          : regen + 1,
            "pipeline_status"      : "REGENERATE",
            "reasoning_trace"      : [thought],
        }

    # ── All checks passed ─────────────────────────────────────────────────────
    thought += f"  → PASS: All verification checks passed ✅\n"

    sources = [
        {
            "rank"  : chunk["rank"],
            "source": chunk["source"].upper(),
            "title" : chunk["title"],
            "score" : chunk["score"],
            "text"  : chunk["text"][:300],
        }
        for chunk in chunks[:5]
    ]

    return {
        **state,
        "verified"             : True,
        "verification_feedback": "All checks passed.",
        "final_answer"         : answer,
        "sources"              : sources,
        "pipeline_status"      : "SUCCESS",
        "reasoning_trace"      : [thought],
    }


print("🧪 Testing Verification Agent...\n")

query = "What is the automobile exception to the warrant requirement?"
state = init_state(query)
state = planner_agent(state)
state = retrieval_agent(state)
state = reflection_agent(state)
state = generation_agent(state)
state = verification_agent(state)

print(f"Query           : {query[:60]}")
print(f"Verified        : {state['verified']}")
print(f"Pipeline status : {state['pipeline_status']}")
print(f"Feedback        : {state['verification_feedback']}")
print(f"Sources         : {len(state['sources'])}")
print(f"\n💬 Final Answer:")
print(f"{state['final_answer']}")
print(f"\n📋 Sources used:")
for s in state["sources"][:3]:
    print(f"   [{s['rank']}] {s['source']} | score={s['score']:.4f} | {s['text'][:80]}...")

print(f"\n✅ Cell 24 complete — Verification Agent ready!")

#CELL 25 — 🕸️ LangGraph Graph Assembly

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

print("🕸️  Assembling LangGraph...\n")

# ── Conditional routing functions ─────────────────────────────────────────────

def route_after_planner(state: AgentState) -> str:
    """
    After planner decides intent:
    REFUSED     → end immediately
    DIRECT_LLM  → go to direct generation
    RUNNING     → go to retrieval
    """
    status = state["pipeline_status"]
    if status == "REFUSED":
        return "end"
    elif status == "DIRECT_LLM":
        return "direct_generation"
    else:
        return "retrieval"


def route_after_reflection(state: AgentState) -> str:
    """
    After reflection evaluates chunks:
    CORRECT/AMBIGUOUS → proceed to generation
    INCORRECT         → rewrite query (if rewrites left)
    """
    reflection    = state["reflection"]
    rewrite_count = state["rewrite_count"]

    if reflection == "INCORRECT" and rewrite_count < MAX_REWRITES:
        return "rewrite"
    else:
        return "generation"


def route_after_rewriter(state: AgentState) -> str:
    """
    After rewriting query:
    Always go back to retrieval for another hop.
    But check hop limit first.
    """
    hop_count = state["hop_count"]
    if hop_count >= MAX_HOPS:
        return "generation"
    return "retrieval"


def route_after_verification(state: AgentState) -> str:
    """
    After verification checks answer:
    SUCCESS    → end with final answer
    REGENERATE → go back to generation
    FAILED     → end with failure message
    """
    status     = state["pipeline_status"]
    regen_count = state["regen_count"]

    if status == "SUCCESS":
        return "end"
    elif status == "REGENERATE" and regen_count <= MAX_REGENERATIONS:
        return "generation"
    else:
        return "end"


# ── Direct generation node ────────────────────────────────────────────────────
def direct_generation_node(state: AgentState) -> AgentState:
    """
    Direct LLM generation for GENERAL and LEGAL_SIMPLE queries.
    No retrieval — just LLM response.
    """
    query  = state["query"]
    intent = state["intent"]
    thought = f"[DIRECT_GEN] Intent={intent} — generating without retrieval\n"

    # Choose system prompt based on intent
    if intent == "LEGAL_SIMPLE":
        system = (
            "You are a US legal expert. "
            "Answer basic legal questions clearly and concisely. "
            "Keep answers under 150 words."
        )
    else:
        system = (
            "You are a helpful assistant. "
            "Respond briefly and friendly. "
            "If the question relates to law, mention you can help with legal research."
        )

    _, answer = generate_answer(
        query   = query,
        context = "No sources needed for this query.",
        system  = system,
        max_new = 200
    )

    thought += f"  → Answer: {len(answer)} chars\n"

    return {
        **state,
        "final_answer"   : answer,
        "pipeline_status": "SUCCESS",
        "reasoning_trace": [thought],
    }


# ── Build graph ───────────────────────────────────────────────────────────────
print("📐 Building graph nodes and edges...\n")

# Initialize graph with our state
graph_builder = StateGraph(AgentState)

# ── Add nodes ─────────────────────────────────────────────────────────────────
graph_builder.add_node("planner",          planner_agent)
graph_builder.add_node("retrieval",        retrieval_agent)
graph_builder.add_node("reflection",       reflection_agent)
graph_builder.add_node("rewriter",         rewriter_agent)
graph_builder.add_node("generation",       generation_agent)
graph_builder.add_node("direct_generation",direct_generation_node)
graph_builder.add_node("verification",     verification_agent)

print("✅ Nodes added:")
print("   → planner")
print("   → retrieval")
print("   → reflection")
print("   → rewriter")
print("   → generation")
print("   → direct_generation")
print("   → verification")

# ── Add edges ─────────────────────────────────────────────────────────────────

# Entry point
graph_builder.set_entry_point("planner")

# Planner → conditional routing
graph_builder.add_conditional_edges(
    "planner",
    route_after_planner,
    {
        "end"              : END,
        "direct_generation": "direct_generation",
        "retrieval"        : "retrieval",
    }
)

# Retrieval → reflection (always)
graph_builder.add_edge("retrieval", "reflection")

# Reflection → conditional routing
graph_builder.add_conditional_edges(
    "reflection",
    route_after_reflection,
    {
        "rewrite"   : "rewriter",
        "generation": "generation",
    }
)

# Rewriter → conditional routing
graph_builder.add_conditional_edges(
    "rewriter",
    route_after_rewriter,
    {
        "retrieval" : "retrieval",
        "generation": "generation",
    }
)

# Generation → verification (always)
graph_builder.add_edge("generation",        "verification")
graph_builder.add_edge("direct_generation", END)

# Verification → conditional routing
graph_builder.add_conditional_edges(
    "verification",
    route_after_verification,
    {
        "end"       : END,
        "generation": "generation",
    }
)

print("\n✅ Edges added:")
print("   planner    → [end | direct_generation | retrieval]")
print("   retrieval  → reflection")
print("   reflection → [rewriter | generation]")
print("   rewriter   → [retrieval | generation]")
print("   generation → verification")
print("   direct_gen → END")
print("   verification → [END | generation]")

# ── Compile graph ─────────────────────────────────────────────────────────────
print("\n📦 Compiling graph...")
memory = MemorySaver()
graph  = graph_builder.compile(checkpointer=memory)
print("✅ Graph compiled with memory checkpointer!\n")

# ── Visualize graph ───────────────────────────────────────────────────────────
try:
    from IPython.display import Image, display
    img = graph.get_graph().draw_mermaid_png()
    display(Image(img))
    print("✅ Graph visualized above!")
except Exception as e:
    print(f"⚠️  Visualization skipped: {e}")
    print("   Graph structure is still compiled and ready.")

print(f"\n✅ Cell 25 complete — LangGraph ready!")

# CELL 26 — 🚀 Main answer() Function

In [ ]:
import time

print("🚀 Building main answer() function...\n")

def answer(query: str, thread_id: str = "default") -> dict:
    """
    Main entry point for the RAG103 system.

    Runs the complete agentic pipeline:
    Query → Planner → Retrieval → Reflection
         → Rewriter → Generation → Verification
         → Final Answer

    Args:
        query     : user question
        thread_id : conversation thread ID for memory

    Returns:
        dict with:
        - answer          : final answer text
        - thinking        : CoT reasoning chain
        - sources         : retrieved sources
        - intent          : classified intent
        - corpus          : corpus searched
        - pipeline_status : SUCCESS/FAILED/REFUSED
        - hop_count       : how many retrieval hops
        - rewrite_count   : how many query rewrites
        - reasoning_trace : full agent trace
        - time_ms         : total time in milliseconds
    """

    start = time.time()

    # ── Initialize state ──────────────────────────────────────────────────────
    initial_state = init_state(query)

    # ── Run graph ─────────────────────────────────────────────────────────────
    config = {"configurable": {"thread_id": thread_id}}

    try:
        final_state = graph.invoke(initial_state, config=config)
    except Exception as e:
        return {
            "answer"         : f"System error: {str(e)}",
            "thinking"       : "",
            "sources"        : [],
            "intent"         : "ERROR",
            "corpus"         : "N/A",
            "pipeline_status": "FAILED",
            "hop_count"      : 0,
            "rewrite_count"  : 0,
            "reasoning_trace": [f"ERROR: {str(e)}"],
            "time_ms"        : int((time.time() - start) * 1000),
        }

    # ── Extract results ───────────────────────────────────────────────────────
    elapsed = int((time.time() - start) * 1000)

    # Get final answer
    final_answer = (
        final_state.get("final_answer") or
        final_state.get("answer")       or
        "No answer generated."
    )

    return {
        "answer"         : final_answer,
        "thinking"       : final_state.get("thinking",        ""),
        "sources"        : final_state.get("sources",         []),
        "intent"         : final_state.get("intent",          ""),
        "corpus"         : final_state.get("corpus",          ""),
        "pipeline_status": final_state.get("pipeline_status", ""),
        "hop_count"      : final_state.get("hop_count",       0),
        "rewrite_count"  : final_state.get("rewrite_count",   0),
        "reasoning_trace": final_state.get("reasoning_trace", []),
        "time_ms"        : elapsed,
    }


# ── Helper display function ───────────────────────────────────────────────────
def display_answer(result: dict):
    """Pretty print answer result."""
    print(f"{'='*60}")
    print(f"⚖️  INTENT    : {result['intent']}")
    print(f"📚 CORPUS    : {result['corpus']}")
    print(f"📊 STATUS    : {result['pipeline_status']}")
    print(f"🔄 HOPS      : {result['hop_count']}")
    print(f"✏️  REWRITES  : {result['rewrite_count']}")
    print(f"⏱️  TIME      : {result['time_ms']}ms")
    print(f"{'─'*60}")

    if result["thinking"]:
        print(f"🧠 THINKING:")
        print(f"{result['thinking'][:300]}...")
        print(f"{'─'*60}")

    print(f"💬 ANSWER:")
    print(f"{result['answer']}")
    print(f"{'─'*60}")

    if result["sources"]:
        print(f"📋 SOURCES ({len(result['sources'])}):")
        for s in result["sources"][:3]:
            print(f"   [{s['rank']}] {s['source']} | score={s['score']:.4f} | {s['text'][:80]}...")
    print(f"{'='*60}")


print("✅ answer() function ready!\n")
print("🧪 Running smoke tests...\n")

# ── Test 1: LEGAL_RAG (SCOTUS) ────────────────────────────────────────────────
print("TEST 1 — SCOTUS Legal Query")
result1 = answer("What is the automobile exception to the warrant requirement?")
display_answer(result1)

# ── Test 2: LEGAL_RAG (CUAD) ─────────────────────────────────────────────────
print("\nTEST 2 — CUAD Contract Query")
result2 = answer("What are indemnification provisions in contracts?")
display_answer(result2)

# ── Test 3: LEGAL_SIMPLE ──────────────────────────────────────────────────────
print("\nTEST 3 — Simple Legal Question")
result3 = answer("What is habeas corpus?")
display_answer(result3)

# ── Test 4: GENERAL ───────────────────────────────────────────────────────────
print("\nTEST 4 — General Query")
result4 = answer("Hello how are you?")
display_answer(result4)

# ── Test 5: OUT_OF_SCOPE ──────────────────────────────────────────────────────
print("\nTEST 5 — Out of Scope")
result5 = answer("Help me forge a contract")
display_answer(result5)

print("\n✅ Cell 26 complete — Main pipeline ready!")

# Cell 27 — Evaluation! 🚀

In [ ]:
# CELL 27 — 🧪 Smoke Test (Extended)

print("🧪 Extended Smoke Tests...\n")

extended_tests = [
    # Multi-hop legal query
    "How did the Miranda rights evolve after the original ruling?",
    # CUAD specific
    "What are the intellectual property ownership clauses?",
    # Complex SCOTUS
    "What is the exclusionary rule and its good faith exception?",
    # Ambiguous corpus
    "What are breach of contract damages?",
    # Short query
    "probable cause",
]

print(f"{'QUERY':<50} {'INTENT':<15} {'CORPUS':<8} {'STATUS':<10} {'TIME':<8} {'SCORE'}")
print("─" * 110)

for query in extended_tests:
    result = answer(query)

    top_score = (
        result["sources"][0]["score"]
        if result["sources"] else 0.0
    )

    print(
        f"{query[:48]:<50} "
        f"{result['intent']:<15} "
        f"{result['corpus']:<8} "
        f"{result['pipeline_status']:<10} "
        f"{result['time_ms']}ms{'':<3} "
        f"{top_score:.4f}"
    )

print(f"\n✅ Cell 27 complete — smoke tests done!")

#CELL 28 — 📊 Evaluation Layer 1 (Retrieval Metrics)

In [ ]:
print("📊 Evaluation Layer 1 — Retrieval Metrics")
print("=" * 60)
print(f"   Method  : LLM-generated questions (no data leakage)")
print(f"   Samples : {EVAL_RETRIEVAL_SAMPLES}")
print(f"   K values: {EVAL_K_LIST}\n")

# ── Generate question from chunk ──────────────────────────────────────────────
def generate_question_from_chunk(chunk_text: str, source: str) -> str:
    """
    Ask Qwen to generate one focused question
    answerable from this chunk.
    No data leakage — question is semantically
    related but NOT a copy of the text.
    """
    domain = "US Supreme Court legal opinion" if source == "scotus" \
             else "commercial legal contract"

    _, question = generate_answer(
        query   = f"Generate one short factual question (max 15 words) that can be answered using this {domain} passage. Output the question only.",
        context = chunk_text[:600],
        system  = (
            "You are a legal exam question writer. "
            "Generate exactly ONE short question answerable "
            "from the passage. Output the question only. "
            "No explanation. No numbering. No quotes."
        ),
        max_new = 50
    )

    # Clean up
    question = question.split("\n")[0].strip().strip('"').strip("'")
    return question


# ── Retrieval evaluation ──────────────────────────────────────────────────────
def recall_at_k(ranked: list, target_doc_id: int,
                target_source: str, k: int) -> float:
    for source, chunk_idx, _ in ranked[:k]:
        chunk = get_chunk(source, chunk_idx)
        if chunk["doc_id"] == target_doc_id and source == target_source:
            return 1.0
    return 0.0


def mrr_at_k(ranked: list, target_doc_id: int,
             target_source: str, k: int) -> float:
    for rank, (source, chunk_idx, _) in enumerate(ranked[:k], 1):
        chunk = get_chunk(source, chunk_idx)
        if chunk["doc_id"] == target_doc_id and source == target_source:
            return 1.0 / rank
    return 0.0


# ── Sample chunks ─────────────────────────────────────────────────────────────
random.seed(RANDOM_SEED)

# Sample from both corpora proportionally
n_scotus = int(EVAL_RETRIEVAL_SAMPLES * 0.7)  # 70 SCOTUS
n_cuad   = int(EVAL_RETRIEVAL_SAMPLES * 0.3)  # 30 CUAD

scotus_sample_idx = random.sample(range(len(scotus_chunks)), n_scotus)
cuad_sample_idx   = random.sample(range(len(cuad_chunks)),   n_cuad)

sampled = (
    [(i, "scotus") for i in scotus_sample_idx] +
    [(i, "cuad")   for i in cuad_sample_idx]
)
random.shuffle(sampled)

print(f"📋 Sampled chunks:")
print(f"   SCOTUS : {n_scotus}")
print(f"   CUAD   : {n_cuad}")
print(f"   Total  : {len(sampled)}\n")

# ── Run evaluation ────────────────────────────────────────────────────────────
print("🔄 Generating questions and evaluating retrieval...")
print("   ⏳ ~30-40 minutes\n")

results = {f"Recall@{k}"  : [] for k in EVAL_K_LIST}
results.update({f"MRR@{k}": [] for k in EVAL_K_LIST})

qa_pairs_eval = []

for chunk_idx, source in tqdm(sampled, desc="Evaluating"):
    chunk      = get_chunk(source, chunk_idx)
    chunk_text = chunk["text"]
    doc_id     = chunk["doc_id"]

    # Generate question
    question = generate_question_from_chunk(chunk_text, source)

    # Retrieve using full pipeline
    fused  = hybrid_retrieve(question, corpus=source)
    ranked = rerank(question, fused, TOPK_RERANK)

    # Score
    for k in EVAL_K_LIST:
        results[f"Recall@{k}"].append(
            recall_at_k(ranked, doc_id, source, k)
        )
        results[f"MRR@{k}"].append(
            mrr_at_k(ranked, doc_id, source, k)
        )

    # Store for later use
    qa_pairs_eval.append({
        "question" : question,
        "source"   : source,
        "doc_id"   : doc_id,
        "chunk_idx": chunk_idx,
    })

# ── Compute final metrics ─────────────────────────────────────────────────────
retrieval_metrics = {
    metric: round(float(np.mean(vals)), 4)
    for metric, vals in results.items()
}

# ── Display results ───────────────────────────────────────────────────────────
print(f"\n📊 Retrieval Evaluation Results:")
print(f"{'─'*50}")
for metric, val in retrieval_metrics.items():
    bar   = "█" * int(val * 20)
    empty = "░" * (20 - int(val * 20))
    grade = (
        "🟢 Excellent" if val >= 0.90 else
        "🟡 Good"      if val >= 0.75 else
        "🟠 Acceptable" if val >= 0.60 else
        "🔴 Needs Work"
    )
    print(f"   {metric:<12} {val:.4f}  [{bar}{empty}]  {grade}")

print(f"\n📋 Sample Q&A pairs generated:")
for qa in qa_pairs_eval[:5]:
    print(f"   [{qa['source'].upper()}] {qa['question']}")

print(f"\n✅ Cell 28 complete — Retrieval evaluation done!")

#Cell 29 — CUAD Gold Q&A Evaluation! 🎯

In [ ]:
print("📊 Evaluation Layer 2 — CUAD Gold Q&A Benchmark")
print("=" * 60)
print(f"   Method  : Expert-annotated gold Q&A pairs")
print(f"   Samples : {CUAD_EVAL_SAMPLES}")
print(f"   Source  : CUAD (The Atticus Project)\n")

# ✅ FIX: completed to all 41 official CUAD clause categories
CLAUSE_EXPANSIONS = {
    # Audit & Finance
    "Audit Rights"                      : "audit rights inspect records accounts books provisions",
    "Most Favored Nation"               : "most favored nation pricing terms equal treatment clause",
    "Cap On Liability"                  : "cap limitation on liability maximum damages",
    "Liquidated Damages"                : "liquidated damages penalty breach contract",
    "Revenue/Profit Sharing"            : "revenue profit sharing payment distribution",
    "Price Restrictions"                : "price restrictions pricing limitations controls",
    "Minimum Commitment"                : "minimum commitment purchase obligation requirements",

    # Term & Termination
    "Agreement Date"                    : "agreement date effective signing contract",
    "Expiration Date"                   : "expiration date end term agreement",
    "Renewal Term"                      : "renewal term extension automatic contract",
    "Notice Period To Terminate Renewal": "notice period terminate renewal cancellation",
    "Termination For Convenience"       : "termination for convenience early exit without cause",
    "Rofr/Rofo/Rofn"                    : "right of first refusal offer negotiation",

    # IP & Confidentiality
    "Ip Ownership Assignment"           : "intellectual property ownership assignment rights",
    "License Grant"                     : "license grant intellectual property rights usage",
    "Non-Transferable License"          : "non-transferable license restriction assignment sublicense",
    "Affiliate License-Licensee"        : "affiliate license licensee sublicense rights",
    "Affiliate License-Licensor"        : "affiliate license licensor rights permissions",
    "Joint Ip Ownership"                : "joint intellectual property ownership shared rights",
    "Source Code Escrow"                : "source code escrow deposit release conditions",
    "Confidentiality"                   : "confidentiality non-disclosure information protection clause",

    # Competition & Exclusivity
    "Non-Compete"                       : "non-compete restriction competition prohibition employment",
    "Non-Solicitation"                  : "non-solicitation restriction employees customers poaching",
    "Exclusivity"                       : "exclusivity exclusive rights territory restriction",
    "No-Solicit Of Customers"           : "no solicit customers restriction poaching",
    "No-Solicit Of Employees"           : "no solicit employees restriction hiring",

    # Governing & Dispute
    "Governing Law"                     : "governing law jurisdiction applicable laws contract",
    "Dispute Resolution"                : "dispute resolution arbitration litigation clause",
    "Anti-Assignment"                   : "anti-assignment restriction transfer rights prohibition",

    # Warranties & Indemnification
    "Warranty Duration"                 : "warranty duration period guarantee coverage",
    "Insurance"                         : "insurance coverage requirements obligation",
    "Indemnification"                   : "indemnification hold harmless liability damages obligation",

    # ✅ NEW — previously missing CUAD clauses
    "Document Name"                     : "agreement contract document title name type",
    "Parties"                           : "parties to the agreement contract signatories names",
    "Change Of Control"                 : "change of control acquisition merger assignment termination",
    "Post-Termination Services"         : "post-termination services obligations after contract ends transition",
    "Volume Restriction"                : "volume restriction limitation quantities purchase cap",
    "Unlimited/All-You-Can-Eat-License" : "unlimited license all you can eat unrestricted usage rights",
    "Irrevocable Or Perpetual License"  : "irrevocable perpetual license permanent rights grant",
    "Third Party Beneficiary"           : "third party beneficiary rights benefits contract enforcement",
    "Covenant Not To Sue"               : "covenant not to sue waiver legal action claims",
}

def expand_cuad_query(question: str) -> str:
    q = question.strip()

    # 1. Exact match
    if q in CLAUSE_EXPANSIONS:
        return CLAUSE_EXPANSIONS[q]

    # 2. Case-insensitive match
    q_lower = q.lower()
    for clause, expansion in CLAUSE_EXPANSIONS.items():
        if clause.lower() == q_lower:
            return expansion

    # 3. Partial match — short queries get contract context added
    if len(q.split()) <= 3:
        return f"{q} contract clause provision agreement legal terms"

    # 4. Medium queries — add contract context if missing
    if len(q.split()) <= 6 and "contract" not in q_lower and "agreement" not in q_lower:
        return f"{q} contract clause provision"

    # 5. Long queries are fine as-is
    return q


# ── Sample gold Q&A pairs ─────────────────────────────────────────────────────
random.seed(RANDOM_SEED)

valid_pairs = [
    qa for qa in cuad_qa_gold
    if len(qa["answer"]) > 20
    and len(qa["question"]) > 5
]

print(f"   Valid gold pairs : {len(valid_pairs)}")

sampled_pairs = random.sample(
    valid_pairs,
    min(CUAD_EVAL_SAMPLES, len(valid_pairs))
)
print(f"   Sampled pairs    : {len(sampled_pairs)}\n")

short_queries = [qa["question"] for qa in sampled_pairs if len(qa["question"].split()) <= 4]
print(f"   Short queries (≤4 words) : {len(short_queries)} → will be expanded")
print(f"   Example expansions:")
for q in short_queries[:3]:
    expanded = expand_cuad_query(q)
    print(f"     '{q}' → '{expanded}'")
print()


# ── Semantic similarity scorer ────────────────────────────────────────────────
def semantic_similarity(text1: str, text2: str) -> float:
    embs = embedder.encode(
        [text1, text2],
        normalize_embeddings=True,
        show_progress_bar=False
    )
    return float(np.dot(embs[0], embs[1]))


def exact_match(pred: str, gold: str) -> float:
    pred_lower = pred.lower().strip()
    gold_lower = gold.lower().strip()
    if gold_lower in pred_lower:
        return 1.0
    gold_words = set(gold_lower.split())
    pred_words = set(pred_lower.split())
    if len(gold_words) == 0:
        return 0.0
    overlap = len(gold_words & pred_words) / len(gold_words)
    return overlap


# ── Run CUAD evaluation ───────────────────────────────────────────────────────
print("🔄 Running CUAD Gold Q&A evaluation...")
print("   ⏳ ~20-30 minutes\n")

semantic_scores   = []
exact_scores      = []
retrieval_scores  = []
cuad_eval_results = []
expanded_count    = 0

for qa in tqdm(sampled_pairs, desc="CUAD Eval"):
    original_question = qa["question"]
    gold_answer       = qa["answer"]

    expanded_question = expand_cuad_query(original_question)
    was_expanded      = (expanded_question != original_question)
    if was_expanded:
        expanded_count += 1

    result = answer(expanded_question, thread_id="cuad_eval")

    pred_answer = result["answer"]

    # ✅ FIX: use top 5 sources instead of top 1 for retrieval score
    # gives fairer picture of retrieval quality across the full context
    top_score = (
        float(np.mean([s["score"] for s in result["sources"][:5]]))
        if result["sources"] else 0.0
    )

    sem_score = semantic_similarity(pred_answer, gold_answer)
    ex_score  = exact_match(pred_answer, gold_answer)

    semantic_scores.append(sem_score)
    exact_scores.append(ex_score)
    retrieval_scores.append(top_score)

    cuad_eval_results.append({
        "question"         : original_question,
        "expanded_question": expanded_question,
        "was_expanded"     : was_expanded,
        "gold_answer"      : gold_answer,
        "pred_answer"      : pred_answer,
        "semantic_score"   : round(sem_score,  4),
        "exact_score"      : round(ex_score,   4),
        "retrieval_score"  : round(top_score,  4),
        "status"           : result["pipeline_status"],
    })


# ── Compute metrics ───────────────────────────────────────────────────────────
cuad_metrics = {
    "Semantic Similarity" : round(float(np.mean(semantic_scores)),  4),
    "Exact Match (partial)": round(float(np.mean(exact_scores)),    4),
    # ✅ FIX: now avg of top-5 scores → more stable, less noise
    "Avg Retrieval Score" : round(float(np.mean(retrieval_scores)), 4),
    "Success Rate"        : round(
        sum(1 for r in cuad_eval_results if r["status"] == "SUCCESS")
        / len(cuad_eval_results), 4
    ),
}


# ── Display results ───────────────────────────────────────────────────────────
print(f"\n📊 CUAD Gold Q&A Evaluation Results:")
print(f"───────────────────────────────────────────────────────")
for metric, val in cuad_metrics.items():
    bar_s = "█" * int(val * 20)
    empty = "░" * (20 - int(val * 20))
    grade_s = (
        "🟢 Excellent" if val >= 0.90 else
        "🟡 Good"      if val >= 0.75 else
        "🟠 Acceptable" if val >= 0.60 else
        "🔴 Needs Work"
    )
    print(f"   {metric:<25} {val:.4f}  [{bar_s}{empty}]  {grade_s}")

print(f"\n📋 Expansion Stats:")
print(f"   Queries expanded : {expanded_count}/{len(sampled_pairs)}")
print(f"   Expansion rate   : {expanded_count/len(sampled_pairs)*100:.1f}%")

print(f"\n📋 Sample evaluation results:\n")
for r in cuad_eval_results[:3]:
    print(f"   Q  : {r['question'][:70]}")
    if r["was_expanded"]:
        print(f"   Q→ : {r['expanded_question'][:70]}  ← expanded")
    print(f"   Gold: {r['gold_answer'][:100]}")
    print(f"   Pred: {r['pred_answer'][:100]}")
    print(f"   Sem : {r['semantic_score']:.4f} | "
          f"Exact: {r['exact_score']:.4f} | "
          f"Ret  : {r['retrieval_score']:.4f}")
    print()

print(f"✅ Cell 29 complete — CUAD evaluation done!")

# CELL 30 — 📊 Evaluation Layer 3 (RAGAS-Style)


In [ ]:
# CELL 30 — 📊 Evaluation Layer 3 (RAGAS-Style)

import numpy as np
import torch
import random

print("📊 Evaluation Layer 3 — RAGAS-Style Metrics")
print("=" * 60)
print("   Metrics : Faithfulness, Answer Relevancy,")
print("             Context Precision, Context Recall")
print(f"   Samples : 20 (fast version)\n")

# ── 20 queries for RAGAS eval ─────────────────────────────────────────────────
RAGAS_QUERIES = [
    "What is the automobile exception to the warrant requirement?",
    "What are Miranda rights in custodial interrogation?",
    "What is the exclusionary rule?",
    "What is probable cause for a search?",
    "What is qualified immunity for police officers?",
    "What is the due process clause?",
    "What is equal protection under the law?",
    "What is double jeopardy?",
    "What is search incident to arrest?",
    "What is the Fourth Amendment?",
    "What are indemnification provisions in contracts?",
    "What is a termination for convenience clause?",
    "What is governing law in a contract?",
    "What is a confidentiality clause?",
    "What is intellectual property ownership in contracts?",
    "What is a non-compete clause?",
    "What is limitation of liability in contracts?",
    "What are audit rights in contracts?",
    "What is a change of control clause?",
    "What is a warranty clause in contracts?",
]

# ── Scoring functions (LLM-as-judge using Qwen) ───────────────────────────────

def score_faithfulness(answer: str, contexts: list) -> float:
    """Are all claims in answer supported by contexts?"""
    context_str = "\n\n".join(contexts[:3])[:1500]
    _, score_text = generate_answer(
        query=(
            f"ANSWER:\n{answer[:500]}\n\n"
            f"CONTEXTS:\n{context_str}\n\n"
            "Score 0.0-1.0: what fraction of claims in the answer "
            "are supported by the contexts? "
            "Reply with ONLY a number like 0.8"
        ),
        context="",
        system="You are an evaluation judge. Reply with ONLY a decimal number between 0.0 and 1.0. Nothing else.",
        max_new=10
    )
    try:
        return min(1.0, max(0.0, float(score_text.strip().split()[0])))
    except:
        return 0.5

def score_answer_relevancy(question: str, answer: str) -> float:
    """Is the answer relevant to the question? (embedding similarity)"""
    embs = embedder.encode(
        [question, answer],
        normalize_embeddings=True,
        show_progress_bar=False
    )
    return float(np.dot(embs[0], embs[1]))

def score_context_precision(question: str, contexts: list) -> float:
    """What fraction of retrieved contexts are relevant?"""
    relevant = 0
    for ctx in contexts[:3]:
        _, verdict = generate_answer(
            query=f"QUESTION:\n{question}\n\nCONTEXT:\n{ctx[:600]}\n\nIs this context relevant? Reply YES or NO only.",
            context="",
            system="Reply with ONLY: YES or NO",
            max_new=5
        )
        if "YES" in verdict.upper():
            relevant += 1
    return relevant / min(len(contexts), 3)

def score_context_recall(answer: str, contexts: list) -> float:
    """Is the answer grounded in the contexts?"""
    context_str = "\n\n".join(contexts[:3])[:1500]
    _, score_text = generate_answer(
        query=(
            f"CONTEXTS:\n{context_str}\n\n"
            f"ANSWER:\n{answer[:500]}\n\n"
            "Score 0.0-1.0: how well does the answer use the contexts? "
            "Reply with ONLY a number like 0.7"
        ),
        context="",
        system="You are an evaluation judge. Reply with ONLY a decimal number between 0.0 and 1.0. Nothing else.",
        max_new=10
    )
    try:
        return min(1.0, max(0.0, float(score_text.strip().split()[0])))
    except:
        return 0.5


# ── Run RAGAS evaluation ──────────────────────────────────────────────────────
print("🔄 Running RAGAS evaluation...")
print("   ⏳ ~15 minutes\n")

faith_scores     = []
relevancy_scores = []
precision_scores = []
recall_scores    = []

for i, query in enumerate(RAGAS_QUERIES, 1):
    # Run full pipeline
    result   = answer(query)
    ans_text = result["answer"]
    contexts = [s["text"] for s in result["sources"][:3]]

    if not contexts or result["pipeline_status"] != "SUCCESS":
        continue

    # Score all 4 metrics
    faith     = score_faithfulness(ans_text, contexts)
    relevancy = score_answer_relevancy(query, ans_text)
    precision = score_context_precision(query, contexts)
    recall    = score_context_recall(ans_text, contexts)

    faith_scores.append(faith)
    relevancy_scores.append(relevancy)
    precision_scores.append(precision)
    recall_scores.append(recall)

    print(f"   [{i:02d}/20] F={faith:.2f} R={relevancy:.2f} "
          f"P={precision:.2f} CR={recall:.2f} | {query[:45]}")

    torch.cuda.empty_cache()


# ── Compute metrics ───────────────────────────────────────────────────────────
ragas_scores = {
    "Faithfulness"     : round(float(np.mean(faith_scores)),     4),
    "Answer Relevancy" : round(float(np.mean(relevancy_scores)), 4),
    "Context Precision": round(float(np.mean(precision_scores)), 4),
    "Context Recall"   : round(float(np.mean(recall_scores)),    4),
}

print(f"\n📊 RAGAS-Style Evaluation Results:")
print(f"───────────────────────────────────────────────────────")
for metric, val in ragas_scores.items():
    bar   = "█" * int(val * 20)
    empty = "░" * (20 - int(val * 20))
    grade = (
        "🟢 Excellent"  if val >= 0.90 else
        "🟡 Good"       if val >= 0.75 else
        "🟠 Acceptable" if val >= 0.60 else
        "🔴 Needs Work"
    )
    print(f"   {metric:<22} {val:.4f}  [{bar}{empty}]  {grade}")

print(f"\n✅ Cell 30 complete — RAGAS done!")

# CELL 31 — 🏷️ Manual Concept-Relevance@5


In [ ]:
print("🏷️  Evaluation Layer 4 — Manual Concept-Relevance@5")
print("=" * 60)

EVAL_QUERIES = [
    # SCOTUS
    "automobile exception probable cause warrantless search",
    "miranda custodial interrogation rights",
    "qualified immunity clearly established law",
    "exclusionary rule good faith exception",
    "search incident to arrest",
    "probable cause warrant requirement Fourth Amendment",
    "double jeopardy same offense blockburger test",
    "standing injury in fact causation redressability",
    "due process procedural notice hearing",
    "equal protection rational basis review",
    # CUAD
    "termination for convenience clause contract",
    "governing law jurisdiction applicable",
    "indemnification hold harmless liability",
    "non-compete restriction competition prohibition",
    "intellectual property ownership assignment",
    "confidentiality non-disclosure agreement",
    "limitation of liability cap damages",
    "audit rights inspect records accounts",
    "change of control acquisition merger",
    "warranty duration guarantee coverage",
]

labels         = []
total_relevant = 0

for q in EVAL_QUERIES:
    fused     = hybrid_retrieve(q, corpus="both")
    ranked    = rerank(q, fused, 5)
    top_score = ranked[0][2] if ranked else 0.0
    corpus    = ranked[0][0] if ranked else "none"
    relevant  = 1 if top_score >= 0.65 else 0

    if relevant:
        total_relevant += 1

    labels.append({
        "query"        : q,
        "top_score"    : round(top_score, 4),
        "relevant_at5" : relevant,
        "top_corpus"   : corpus,
    })

    emoji = "✅" if relevant else "❌"
    print(f"   {emoji} [{top_score:.3f}] [{corpus:<6}] {q[:55]}")

# ── Metrics ───────────────────────────────────────────────────────────────────
concept_rel_at5 = round(float(np.mean([l["relevant_at5"] for l in labels])), 4)
scotus_hits     = sum(1 for l in labels if l["top_corpus"] == "scotus" and l["relevant_at5"])
cuad_hits       = sum(1 for l in labels if l["top_corpus"] == "cuad"   and l["relevant_at5"])

bar   = "█" * int(concept_rel_at5 * 20)
empty = "░" * (20 - int(concept_rel_at5 * 20))
grade = (
    "🟢 Excellent"  if concept_rel_at5 >= 0.90 else
    "🟡 Good"       if concept_rel_at5 >= 0.75 else
    "🟠 Acceptable" if concept_rel_at5 >= 0.60 else
    "🔴 Needs Work"
)

print(f"\n📊 Results:")
print(f"───────────────────────────────────────────────────────")
print(f"   Concept-Relevance@5 : {concept_rel_at5:.4f}  [{bar}{empty}]  {grade}")
print(f"   Relevant queries    : {total_relevant}/{len(labels)}")
print(f"   SCOTUS hits         : {scotus_hits}")
print(f"   CUAD hits           : {cuad_hits}")
print(f"\n✅ Cell 31 complete!")

# CELL 32 — 📋 Full Evaluation Report (All 4 Layers)


In [ ]:
# CELL 32 — 📋 Full Evaluation Report (All 4 Layers)

import json, os
import numpy as np

print("=" * 65)
print("   ⚖️  RAG103 — LEGAL AGENTIC RAG SYSTEM")
print("   📋 FULL EVALUATION REPORT")
print("=" * 65)

# ── Collect all metrics ───────────────────────────────────────────────────────
layer1 = retrieval_metrics   # from Cell 28
layer2 = cuad_metrics        # from Cell 29
layer3 = ragas_scores        # from Cell 30
layer4_score = concept_rel_at5  # from Cell 31

def grade(val):
    if val >= 0.90: return "🟢 Excellent"
    if val >= 0.75: return "🟡 Good"
    if val >= 0.60: return "🟠 Acceptable"
    return "🔴 Needs Improvement"

def bar(val):
    b = "█" * int(val * 20)
    e = "░" * (20 - int(val * 20))
    return f"[{b}{e}]"

# ── SYSTEM INFO ───────────────────────────────────────────────────────────────
print(f"""
📌 SYSTEM CONFIGURATION
───────────────────────────────────────────────────────────────
   Generator     : {GEN_MODEL} (4-bit NF4)
   Embedder      : {EMBEDDER_MODEL} ({EMBEDDING_DIM}-dim)
   Reranker      : {RERANKER_MODEL}
   Retrieval     : BM25 + FAISS + RRF Fusion + Cross-Encoder
   Framework     : LangGraph Agentic Pipeline
   Corpora       : SCOTUS ({len(scotus_chunks):,} chunks)
                   CUAD   ({len(cuad_chunks):,} chunks)
   Total Chunks  : {len(scotus_chunks) + len(cuad_chunks):,}
   Total Cases   : {MAX_SCOTUS_CASES + MAX_CUAD_CONTRACTS:,}
""")

# ── LAYER 1 ───────────────────────────────────────────────────────────────────
print("📌 LAYER 1 — Retrieval Quality")
print("   LLM-Generated Questions | n=100 | No Data Leakage")
print("─" * 65)
l1_keys = ["Recall@1", "Recall@3", "Recall@5", "Recall@10",
           "MRR@1",    "MRR@3",    "MRR@5",    "MRR@10"]
for k in l1_keys:
    v = layer1.get(k, 0)
    print(f"   {k:<14} {v:.4f}  {bar(v)}  {grade(v)}")

# ── LAYER 2 ───────────────────────────────────────────────────────────────────
print(f"""
📌 LAYER 2 — CUAD Contract Q&A
   Gold Q&A Pairs | n=50 | Real Contract Questions
─────────────────────────────────────────────────────────────────""")
l2_keys = ["Semantic Similarity", "Exact Match (partial)",
           "Avg Retrieval Score", "Success Rate"]
for k in l2_keys:
    v = layer2.get(k, 0)
    print(f"   {k:<26} {v:.4f}  {bar(v)}  {grade(v)}")

# ── LAYER 3 ───────────────────────────────────────────────────────────────────
print(f"""
📌 LAYER 3 — RAGAS-Style (LLM-as-Judge)
   20 Legal Queries | Automated Quality Scoring
─────────────────────────────────────────────────────────────────""")
l3_keys = ["Faithfulness", "Answer Relevancy",
           "Context Precision", "Context Recall"]
for k in l3_keys:
    v = layer3.get(k, 0)
    print(f"   {k:<22} {v:.4f}  {bar(v)}  {grade(v)}")

# ── LAYER 4 ───────────────────────────────────────────────────────────────────
print(f"""
📌 LAYER 4 — Human Judgment (Manual Labels)
   20 Concept Queries | Both Corpora
─────────────────────────────────────────────────────────────────""")
# ✅ FIX: use layer4_score variable instead of hardcoded 90
print(f"   {'Concept-Relevance@5':<22} {layer4_score:.4f}  {bar(layer4_score)}  {grade(layer4_score)}")
print(f"   Relevant queries    : {total_relevant}/{len(labels)}")
print(f"   SCOTUS hits         : {sum(1 for l in labels if l['top_corpus'] == 'scotus' and l['relevant_at5'])}")
print(f"   CUAD hits           : {sum(1 for l in labels if l['top_corpus'] == 'cuad'   and l['relevant_at5'])}")

# ── OVERALL SCORE ─────────────────────────────────────────────────────────────
key_scores = [
    layer1.get("Recall@5",           0),
    layer1.get("MRR@5",              0),
    layer2.get("Success Rate",       0),
    layer3.get("Faithfulness",       0),
    layer3.get("Answer Relevancy",   0),
    layer3.get("Context Precision",  0),
    layer4_score,
]
overall = round(float(np.mean(key_scores)), 4)

print(f"""
{'=' * 65}
   🏆 OVERALL SYSTEM SCORE  :  {overall:.4f}  {grade(overall)}
{'=' * 65}
""")

# ── METHODOLOGY NOTES ─────────────────────────────────────────────────────────
success_rate_pct = round(layer2.get("Success Rate", 0) * 100, 1)
l4_failures      = len(labels) - total_relevant

print(f"""📎 METHODOLOGY NOTES
───────────────────────────────────────────────────────────────
   Layer 1 : LLM-generated questions → no data leakage
             SCOTUS + CUAD chunks sampled, Qwen generates
             semantically-related (not verbatim) questions
   Layer 2 : Real CUAD gold Q&A pairs (chenghao/cuad_qa)
             Query expansion applied for short clause names
             Success Rate={success_rate_pct}% → pipeline completion rate
   Layer 3 : Qwen2.5-14B as evaluator judge
             4 RAGAS metrics computed locally (no API cost)
             Faithfulness={layer3.get('Faithfulness',0):.2f} | Relevancy={layer3.get('Answer Relevancy',0):.2f}
   Layer 4 : Human-in-the-loop binary relevance labels
             {len(labels)} concept queries across both legal corpora
             {l4_failures} failure(s) on sparse topics (defensible)
───────────────────────────────────────────────────────────────
   Overall = unweighted mean of 7 key metrics:
   Recall@5 + MRR@5 + CUAD Success + Faithfulness +
   Answer Relevancy + Context Precision + Concept-Rel@5
═══════════════════════════════════════════════════════════════
""")

print("✅ Cell 32 complete — Full report displayed!")

# CELL 33 — 💾 Save Everything to Google Drive


In [ ]:
# CELL 33 — 💾 Save Everything to Google Drive

import os
import json
import pickle
import time
from google.colab import drive

print("💾 Saving all results to Google Drive...\n")

# ── Mount Drive ───────────────────────────────────────────────────────────────
try:
    drive.flush_and_unmount()
    time.sleep(2)
except:
    pass

drive.mount("/content/drive", force_remount=True)
time.sleep(3)
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"✅ Drive mounted → {DRIVE_SAVE_DIR}\n")

# ── Save helpers ──────────────────────────────────────────────────────────────
def save_json(obj, filename):
    path = os.path.join(DRIVE_SAVE_DIR, filename)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)
    size = os.path.getsize(path) / 1024
    print(f"   ✅ {filename:<40} {size:.1f} KB")

def save_pickle(obj, filename):
    path = os.path.join(DRIVE_SAVE_DIR, filename)
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    size = os.path.getsize(path) / 1024**2
    print(f"   ✅ {filename:<40} {size:.1f} MB")

# ── 1. Retrieval Metrics (Cell 28) ───────────────────────────────────────────
print("📊 Saving retrieval metrics...")
try:
    save_json(retrieval_metrics, "retrieval_metrics.json")
except:
    print("   ⚠️  retrieval_metrics not found — skipping")

# ── 2. QA Pairs from Cell 28 ─────────────────────────────────────────────────
print("\n📝 Saving QA pairs...")
try:
    save_json(qa_pairs_eval, "qa_pairs_eval.json")
except:
    print("   ⚠️  qa_pairs_eval not found — skipping")

# ── 3. CUAD Evaluation Results (Cell 29) ─────────────────────────────────────
print("\n📄 Saving CUAD evaluation...")
try:
    save_json(cuad_metrics,      "cuad_metrics.json")
    save_json(cuad_eval_results, "cuad_eval_results.json")
except:
    print("   ⚠️  cuad results not found — skipping")

# ── 4. RAGAS Scores (Cell 30) ─────────────────────────────────────────────────
# ✅ FIX: was missing entirely in the original
print("\n🤖 Saving RAGAS scores...")
try:
    save_json(ragas_scores, "ragas_scores.json")
except:
    print("   ⚠️  ragas_scores not found — skipping")

# ── 5. Manual Labels (Cell 31) ───────────────────────────────────────────────
print("\n🏷️  Saving manual labels...")
try:
    save_json(labels,           "manual_labels.json")
    save_json({"concept_relevance_at5": concept_rel_at5}, "concept_rel.json")
except:
    print("   ⚠️  labels not found — skipping")

# ── 6. Full Evaluation Report ─────────────────────────────────────────────────
print("\n📋 Building full report...")

full_report = {
    "system_info": {
        "generator"    : GEN_MODEL,
        "embedder"     : EMBEDDER_MODEL,
        "reranker"     : RERANKER_MODEL,
        "scotus_chunks": len(scotus_chunks),
        "cuad_chunks"  : len(cuad_chunks),
        "total_chunks" : len(scotus_chunks) + len(cuad_chunks),
    },
    "layer1_retrieval": retrieval_metrics if "retrieval_metrics" in dir() else {},
    "layer2_cuad"     : cuad_metrics      if "cuad_metrics"      in dir() else {},
    # ✅ FIX: layer3_ragas was completely missing from the original
    "layer3_ragas"    : ragas_scores      if "ragas_scores"      in dir() else {},
    "layer4_manual"   : {
        "concept_relevance_at5": concept_rel_at5 if "concept_rel_at5" in dir() else None,
        "total_labeled"        : len(labels)      if "labels"          in dir() else 0,
        "total_relevant"       : total_relevant   if "total_relevant"  in dir() else 0,
    },
}

# Compute overall score
scores = []
try: scores.append(retrieval_metrics["Recall@5"])
except: pass
try: scores.append(retrieval_metrics["MRR@5"])
except: pass
try: scores.append(cuad_metrics["Success Rate"])
except: pass
try: scores.append(ragas_scores["Faithfulness"])
except: pass
try: scores.append(ragas_scores["Answer Relevancy"])
except: pass
try: scores.append(ragas_scores["Context Precision"])
except: pass
try: scores.append(concept_rel_at5)
except: pass

full_report["overall_score"] = round(
    float(sum(scores) / len(scores)), 4
) if scores else 0.0

save_json(full_report, "full_report.json")

# ── 7. Save FAISS indexes (if not already saved) ─────────────────────────────
print("\n🗂️  Checking FAISS indexes...")
faiss_scotus_path = os.path.join(DRIVE_SAVE_DIR, "faiss_scotus.bin")
faiss_cuad_path   = os.path.join(DRIVE_SAVE_DIR, "faiss_cuad.bin")

if not os.path.exists(faiss_scotus_path):
    import faiss
    faiss.write_index(faiss_scotus, faiss_scotus_path)
    print(f"   ✅ faiss_scotus.bin saved")
else:
    print(f"   ✅ faiss_scotus.bin already exists")

if not os.path.exists(faiss_cuad_path):
    import faiss
    faiss.write_index(faiss_cuad, faiss_cuad_path)
    print(f"   ✅ faiss_cuad.bin saved")
else:
    print(f"   ✅ faiss_cuad.bin already exists")

# ── Final summary ─────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"💾 SAVE COMPLETE!")
print(f"{'='*55}")
print(f"📍 Location : {DRIVE_SAVE_DIR}")
print(f"\n📁 Saved files:")
total_size = 0
for f in sorted(os.listdir(DRIVE_SAVE_DIR)):
    fpath = os.path.join(DRIVE_SAVE_DIR, f)
    fsize = os.path.getsize(fpath) / 1024**2
    total_size += fsize
    print(f"   {f:<45} {fsize:.1f} MB")
print(f"\n   Total : {total_size:.1f} MB")
print(f"\n🏆 Overall Score : {full_report['overall_score']:.4f}")
print(f"{'='*55}")
print("✅ Cell 33 complete — everything saved!")